# Diffusion Models: A Comprehensive Guide

## 1. Introduction

Diffusion models are a class of **deep generative models** that learn to generate data by reversing a gradual noising process. They have emerged as the state-of-the-art in generative modeling, surpassing GANs in image quality and diversity while offering stable training dynamics.

### Core Intuition

The fundamental idea is elegantly simple:
1. **Forward Process (Diffusion)**: Gradually destroy data structure by adding noise over many steps until the data becomes pure Gaussian noise.
2. **Reverse Process (Denoising)**: Learn a neural network to reverse each noising step, thereby generating data from noise.

This is analogous to watching a drop of ink dissolve in water (forward process) and then learning to reconstruct the ink drop from the dispersed molecules (reverse process).

### Historical Context

| Year | Milestone | Authors |
| --- | --- | --- |
| 2015 | Deep Unsupervised Learning using Nonequilibrium Thermodynamics | Sohl-Dickstein et al. |
| 2019 | Generative Modeling by Estimating Gradients of Data Distribution | Song & Ermon |
| 2020 | Denoising Diffusion Probabilistic Models (DDPM) | Ho et al. |
| 2020 | Score-Based Generative Modeling through SDEs | Song et al. |
| 2021 | Denoising Diffusion Implicit Models (DDIM) | Song et al. |
| 2022 | Latent Diffusion Models / Stable Diffusion | Rombach et al. |
| 2022 | Classifier-Free Diffusion Guidance | Ho & Salimans |

### Taxonomy of Diffusion Models

```
Diffusion Models
├── Denoising Diffusion Probabilistic Models (DDPM)
│   ├── Discrete-time formulation
│   └── Variance learning variants (Improved DDPM)
├── Score-Based Generative Models (SGM)
│   ├── Score Matching with Langevin Dynamics (SMLD)
│   └── Noise Conditional Score Networks (NCSN)
├── Stochastic Differential Equations (Score SDE)
│   ├── Variance Preserving SDE (VP-SDE)
│   ├── Variance Exploding SDE (VE-SDE)
│   └── Probability Flow ODE
├── Accelerated Sampling
│   ├── DDIM (Deterministic)
│   ├── DPM-Solver
│   └── Consistency Models
├── Latent Diffusion Models (LDM)
│   └── Stable Diffusion
└── Conditional Diffusion Models
    ├── Classifier Guidance
    └── Classifier-Free Guidance
```

## 2. Mathematical Preliminaries

Before diving into specific algorithms, we establish the mathematical tools that underpin all diffusion models.

### 2.1 Gaussian Distribution & Reparameterization

A multivariate Gaussian distribution with mean $$\mu$$ and covariance $$\Sigma$$ has density:

$$p(x) = \frac{1}{(2\pi)^{d/2} |\Sigma|^{1/2}} \exp\left(-\frac{1}{2}(x - \mu)^T \Sigma^{-1} (x - \mu)\right)$$

**Reparameterization Trick**: Instead of sampling $$x \sim \mathcal{N}(\mu, \sigma^2 I)$$, we write:

$$x = \mu + \sigma \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

This allows gradients to flow through the sampling operation during training.

**Key Property**: If $$x \sim \mathcal{N}(\mu_1, \sigma_1^2)$$ and we add noise: $$y = \sqrt{a} \cdot x + \sqrt{1-a} \cdot \epsilon$$, then:

$$y \sim \mathcal{N}(\sqrt{a} \cdot \mu_1, a \cdot \sigma_1^2 + (1-a) \cdot I)$$

### 2.2 KL Divergence

The Kullback-Leibler divergence measures how one probability distribution $$q$$ differs from a reference $$p$$:

$$D_{KL}(q \| p) = \mathbb{E}_{x \sim q}\left[\log \frac{q(x)}{p(x)}\right]$$

For two Gaussians $$q = \mathcal{N}(\mu_1, \sigma_1^2)$$ and $$p = \mathcal{N}(\mu_2, \sigma_2^2)$$ in 1D:

$$D_{KL}(q \| p) = \log \frac{\sigma_2}{\sigma_1} + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\sigma_2^2} - \frac{1}{2}$$

### 2.3 Evidence Lower Bound (ELBO)

For a latent variable model with observed $$x$$ and latent $$z$$, the log-likelihood decomposes as:

$$\log p(x) = \underbrace{\mathbb{E}_{q(z|x)}\left[\log \frac{p(x,z)}{q(z|x)}\right]}_{\text{ELBO } \mathcal{L}} + \underbrace{D_{KL}(q(z|x) \| p(z|x))}_{\geq 0}$$

Since $$D_{KL} \geq 0$$, the ELBO is a lower bound: $$\log p(x) \geq \mathcal{L}$$. Maximizing the ELBO is equivalent to jointly:
* Maximizing the reconstruction likelihood
* Minimizing the KL divergence between approximate and true posterior

### 2.4 Score Function

The **score function** of a distribution $$p(x)$$ is the gradient of its log-density:

$$s(x) = \nabla_x \log p(x)$$

Key properties:
* Points toward regions of higher probability
* Does not require knowing the normalizing constant (it cancels in the gradient)
* Forms the basis of score-based generative models

### 2.5 Markov Chains and Transition Kernels

A Markov chain is a sequence of random variables where each depends only on the previous:

$$p(x_0, x_1, \ldots, x_T) = p(x_0) \prod_{t=1}^{T} p(x_t | x_{t-1})$$

Each $$p(x_t | x_{t-1})$$ is called a **transition kernel**. In diffusion models, these are typically Gaussian:

$$p(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1-\beta_t} \cdot x_{t-1}, \beta_t I)$$

### 2.6 Langevin Dynamics

Langevin dynamics is an MCMC method that uses the score function to sample from a distribution:

$$x_{k+1} = x_k + \frac{\eta}{2} \nabla_x \log p(x_k) + \sqrt{\eta} \cdot z_k, \quad z_k \sim \mathcal{N}(0, I)$$

As $$\eta \to 0$$ and the number of steps $$K \to \infty$$, $$x_K$$ converges to a sample from $$p(x)$$.

In [0]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from typing import Tuple, Optional
import math

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 3. Denoising Diffusion Probabilistic Models (DDPM)

**Paper**: Ho, Jain, Abbeel (2020) — "Denoising Diffusion Probabilistic Models"

**Industrial Example**: DDPM forms the backbone of **DALL·E 2** (OpenAI) for text-to-image generation, **Imagen** (Google) for photorealistic image synthesis, and **Make-A-Video** (Meta) for video generation from text.

---

### 3.1 Forward Diffusion Process

The forward process defines a Markov chain that gradually adds Gaussian noise to data $$x_0 \sim q(x_0)$$ over $$T$$ timesteps:

$$q(x_t | x_{t-1}) = \mathcal{N}(x_t; \sqrt{1 - \beta_t} \cdot x_{t-1}, \beta_t I)$$

where $$\{\beta_t\}_{t=1}^T$$ is the **variance schedule** (typically $$\beta_1 = 10^{-4}$$ to $$\beta_T = 0.02$$).

The full forward process joint distribution:

$$q(x_{1:T} | x_0) = \prod_{t=1}^{T} q(x_t | x_{t-1})$$

### 3.2 Closed-Form Marginals

A crucial property: we can sample $$x_t$$ at **any** timestep directly from $$x_0$$ without iterating through intermediate steps.

Define $$\alpha_t = 1 - \beta_t$$ and $$\bar{\alpha}_t = \prod_{s=1}^{t} \alpha_s$$. Then:

$$q(x_t | x_0) = \mathcal{N}(x_t; \sqrt{\bar{\alpha}_t} \cdot x_0, (1 - \bar{\alpha}_t) I)$$

Using the reparameterization trick:

$$x_t = \sqrt{\bar{\alpha}_t} \cdot x_0 + \sqrt{1 - \bar{\alpha}_t} \cdot \epsilon, \quad \epsilon \sim \mathcal{N}(0, I)$$

**Proof** (by induction): 
* Base case: $$x_1 = \sqrt{\alpha_1} \cdot x_0 + \sqrt{1 - \alpha_1} \cdot \epsilon_1$$
* Inductive step: Given $$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot x_0 + \sqrt{1 - \bar{\alpha}_{t-1}} \cdot \bar{\epsilon}_{t-1}$$
* Then: $$x_t = \sqrt{\alpha_t} \cdot x_{t-1} + \sqrt{1 - \alpha_t} \cdot \epsilon_t$$
* Substituting: $$x_t = \sqrt{\alpha_t \bar{\alpha}_{t-1}} \cdot x_0 + \sqrt{\alpha_t(1 - \bar{\alpha}_{t-1})} \cdot \bar{\epsilon}_{t-1} + \sqrt{1-\alpha_t} \cdot \epsilon_t$$
* Since the sum of independent Gaussians is Gaussian with variances adding: $$\text{Var} = \alpha_t(1-\bar{\alpha}_{t-1}) + (1-\alpha_t) = 1 - \bar{\alpha}_t$$ ✓

### 3.3 Noise Schedule Strategies

| Schedule | Formula | Use Case |
| --- | --- | --- |
| Linear | $$\beta_t = \beta_1 + \frac{t-1}{T-1}(\beta_T - \beta_1)$$ | Original DDPM |
| Cosine | $$\bar{\alpha}_t = \frac{f(t)}{f(0)}, f(t) = \cos\left(\frac{t/T + s}{1+s} \cdot \frac{\pi}{2}\right)^2$$ | Improved DDPM |
| Sigmoid | $$\beta_t = \sigma(-6 + 12t/T) \cdot (\beta_T - \beta_1) + \beta_1$$ | Stable Diffusion 3 |

In [0]:
# =============================================================================
# DDPM Forward Process: Visualization of Progressive Noise Addition
# =============================================================================

def linear_beta_schedule(T: int, beta_start: float = 1e-4, beta_end: float = 0.02) -> torch.Tensor:
    """Linear variance schedule as proposed in the original DDPM paper."""
    return torch.linspace(beta_start, beta_end, T)

def cosine_beta_schedule(T: int, s: float = 0.008) -> torch.Tensor:
    """Cosine schedule from Improved DDPM (Nichol & Dhariwal, 2021).
    Provides more gradual noise addition, preserving signal longer."""
    steps = torch.arange(T + 1, dtype=torch.float64)
    f_t = torch.cos(((steps / T) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = f_t / f_t[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999).float()

def compute_schedule_params(betas: torch.Tensor) -> dict:
    """Precompute all schedule-dependent parameters."""
    alphas = 1.0 - betas
    alphas_cumprod = torch.cumprod(alphas, dim=0)
    alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
    
    return {
        'betas': betas,
        'alphas': alphas,
        'alphas_cumprod': alphas_cumprod,
        'alphas_cumprod_prev': alphas_cumprod_prev,
        'sqrt_alphas_cumprod': torch.sqrt(alphas_cumprod),
        'sqrt_one_minus_alphas_cumprod': torch.sqrt(1.0 - alphas_cumprod),
        'sqrt_recip_alphas': torch.sqrt(1.0 / alphas),
        'posterior_variance': betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod),
        'posterior_mean_coef1': betas * torch.sqrt(alphas_cumprod_prev) / (1.0 - alphas_cumprod),
        'posterior_mean_coef2': (1.0 - alphas_cumprod_prev) * torch.sqrt(alphas) / (1.0 - alphas_cumprod),
    }

# Demonstrate forward process on 2D data (Swiss Roll)
from sklearn.datasets import make_swiss_roll

# Generate 2D Swiss Roll dataset
data_3d, _ = make_swiss_roll(n_samples=5000, noise=0.5)
data_2d = torch.tensor(data_3d[:, [0, 2]], dtype=torch.float32)  # Take x, z dimensions
data_2d = (data_2d - data_2d.mean(0)) / data_2d.std(0)  # Normalize

# Setup schedule
T = 1000
betas = linear_beta_schedule(T)
params = compute_schedule_params(betas)

# Forward diffusion at selected timesteps
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
timesteps_to_show = [0, 50, 100, 200, 500, 700, 800, 900, 950, 999]

for idx, t in enumerate(timesteps_to_show):
    row, col = idx // 5, idx % 5
    if t == 0:
        x_t = data_2d
    else:
        # Direct sampling using closed-form marginal: x_t = sqrt(alpha_bar_t) * x_0 + sqrt(1-alpha_bar_t) * eps
        eps = torch.randn_like(data_2d)
        x_t = params['sqrt_alphas_cumprod'][t] * data_2d + params['sqrt_one_minus_alphas_cumprod'][t] * eps
    
    axes[row, col].scatter(x_t[:, 0].numpy(), x_t[:, 1].numpy(), s=1, alpha=0.5, c='steelblue')
    axes[row, col].set_title(f't = {t}\n$\\bar{{\\alpha}}_t$ = {params["alphas_cumprod"][t]:.4f}', fontsize=11)
    axes[row, col].set_xlim(-4, 4)
    axes[row, col].set_ylim(-4, 4)
    axes[row, col].set_aspect('equal')

plt.suptitle('Forward Diffusion Process: Swiss Roll → Gaussian Noise', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Plot noise schedule comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

betas_linear = linear_beta_schedule(T)
betas_cosine = cosine_beta_schedule(T)

params_linear = compute_schedule_params(betas_linear)
params_cosine = compute_schedule_params(betas_cosine)

axes[0].plot(betas_linear.numpy(), label='Linear', color='blue')
axes[0].plot(betas_cosine.numpy(), label='Cosine', color='red')
axes[0].set_title('Variance Schedule β_t')
axes[0].set_xlabel('Timestep t')
axes[0].legend()

axes[1].plot(params_linear['alphas_cumprod'].numpy(), label='Linear', color='blue')
axes[1].plot(params_cosine['alphas_cumprod'].numpy(), label='Cosine', color='red')
axes[1].set_title('Cumulative Product $\\bar{\\alpha}_t$ (Signal Retention)')
axes[1].set_xlabel('Timestep t')
axes[1].legend()

snr_linear = params_linear['alphas_cumprod'] / (1 - params_linear['alphas_cumprod'])
snr_cosine = params_cosine['alphas_cumprod'] / (1 - params_cosine['alphas_cumprod'])
axes[2].plot(torch.log10(snr_linear).numpy(), label='Linear', color='blue')
axes[2].plot(torch.log10(snr_cosine).numpy(), label='Cosine', color='red')
axes[2].set_title('Signal-to-Noise Ratio (log₁₀ SNR)')
axes[2].set_xlabel('Timestep t')
axes[2].legend()

plt.suptitle('Noise Schedule Comparison', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 3.4 Reverse Process

The reverse process is a learned Markov chain that starts from noise $$x_T \sim \mathcal{N}(0, I)$$ and iteratively denoises:

$$p_\theta(x_{t-1} | x_t) = \mathcal{N}(x_{t-1}; \mu_\theta(x_t, t), \Sigma_\theta(x_t, t))$$

**Key Insight**: The true reverse posterior $$q(x_{t-1} | x_t, x_0)$$ is tractable and Gaussian when conditioned on $$x_0$$:

$$q(x_{t-1} | x_t, x_0) = \mathcal{N}(x_{t-1}; \tilde{\mu}_t(x_t, x_0), \tilde{\beta}_t I)$$

where:

$$\tilde{\mu}_t(x_t, x_0) = \frac{\sqrt{\bar{\alpha}_{t-1}} \beta_t}{1 - \bar{\alpha}_t} x_0 + \frac{\sqrt{\alpha_t}(1 - \bar{\alpha}_{t-1})}{1 - \bar{\alpha}_t} x_t$$

$$\tilde{\beta}_t = \frac{1 - \bar{\alpha}_{t-1}}{1 - \bar{\alpha}_t} \beta_t$$

### 3.5 Training Objective Derivation

The variational lower bound (ELBO) for diffusion models decomposes into:

$$\mathcal{L} = \underbrace{D_{KL}(q(x_T|x_0) \| p(x_T))}_{L_T \text{ (constant)}} + \sum_{t=2}^{T} \underbrace{D_{KL}(q(x_{t-1}|x_t, x_0) \| p_\theta(x_{t-1}|x_t))}_{L_{t-1}} - \underbrace{\log p_\theta(x_0|x_1)}_{L_0}$$

### 3.6 Three Equivalent Parameterizations

Since $$x_0 = \frac{1}{\sqrt{\bar{\alpha}_t}}(x_t - \sqrt{1-\bar{\alpha}_t} \epsilon)$$, we can parameterize the network to predict:

| Parameterization | Network Predicts | Mean Formula |
| --- | --- | --- |
| $$\epsilon$$-prediction | $$\epsilon_\theta(x_t, t) \approx \epsilon$$ | $$\mu_\theta = \frac{1}{\sqrt{\alpha_t}}\left(x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}}\epsilon_\theta(x_t, t)\right)$$ |
| $$x_0$$-prediction | $$x_{0,\theta}(x_t, t) \approx x_0$$ | $$\mu_\theta = \frac{\sqrt{\bar{\alpha}_{t-1}}\beta_t}{1-\bar{\alpha}_t} x_{0,\theta} + \frac{\sqrt{\alpha_t}(1-\bar{\alpha}_{t-1})}{1-\bar{\alpha}_t} x_t$$ |
| $$v$$-prediction | $$v_\theta(x_t, t) \approx \sqrt{\bar{\alpha}_t}\epsilon - \sqrt{1-\bar{\alpha}_t}x_0$$ | Derived from $$v$$ |

### 3.7 Simplified Loss (Ho et al., 2020)

The key simplification: ignore the weighting terms and use a simple MSE loss:

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, x_0, \epsilon}\left[\| \epsilon - \epsilon_\theta(x_t, t) \|^2\right]$$

where $$t \sim \text{Uniform}(1, T)$$, $$x_0 \sim q(x_0)$$, $$\epsilon \sim \mathcal{N}(0, I)$$, and $$x_t = \sqrt{\bar{\alpha}_t} x_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$$.

---

### Algorithm 1: DDPM Training

```
repeat:
    x_0 ~ q(x_0)                           # Sample data
    t ~ Uniform({1, ..., T})               # Sample timestep
    ε ~ N(0, I)                            # Sample noise
    x_t = sqrt(ā_t) * x_0 + sqrt(1-ā_t) * ε   # Create noisy sample
    Take gradient step on: ||ε - ε_θ(x_t, t)||^2
until converged
```

### Algorithm 2: DDPM Sampling

```
x_T ~ N(0, I)
for t = T, T-1, ..., 1:
    z ~ N(0, I) if t > 1, else z = 0
    x_{t-1} = (1/sqrt(α_t)) * (x_t - (β_t/sqrt(1-ā_t)) * ε_θ(x_t, t)) + sqrt(β_t) * z
return x_0
```

In [0]:
# =============================================================================
# DDPM Full Implementation with a Simple UNet-style Architecture
# =============================================================================

# --- Sinusoidal Time Embedding (Transformer-style positional encoding) ---
class SinusoidalPositionEmbedding(nn.Module):
    """Encodes timestep t into a continuous vector using sinusoidal functions.
    Similar to positional encoding in Transformers."""
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
    
    def forward(self, t: torch.Tensor) -> torch.Tensor:
        device = t.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = t[:, None] * embeddings[None, :]
        embeddings = torch.cat([embeddings.sin(), embeddings.cos()], dim=-1)
        return embeddings


# --- Residual Block with Time Conditioning ---
class ResidualBlock(nn.Module):
    """A residual block conditioned on time embedding."""
    def __init__(self, in_dim: int, out_dim: int, time_dim: int):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.SiLU(),  # Swish activation (preferred in diffusion models)
        )
        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_dim, out_dim),
        )
        self.block2 = nn.Sequential(
            nn.Linear(out_dim, out_dim),
            nn.SiLU(),
        )
        self.residual = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
    
    def forward(self, x: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        h = self.block1(x)
        h = h + self.time_mlp(t_emb)  # Add time conditioning
        h = self.block2(h)
        return h + self.residual(x)  # Skip connection


# --- Noise Prediction Network (Epsilon-theta) ---
class NoisePredictor(nn.Module):
    """A simple MLP-based noise prediction network for 2D data.
    For images, this would be a UNet with attention layers."""
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, time_dim: int = 128):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.res_blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, hidden_dim, time_dim),
            ResidualBlock(hidden_dim, hidden_dim, time_dim),
            ResidualBlock(hidden_dim, hidden_dim, time_dim),
            ResidualBlock(hidden_dim, hidden_dim, time_dim),
        ])
        self.output_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        """Predict the noise added to x at timestep t.
        Args:
            x: Noisy data [batch_size, data_dim]
            t: Timestep [batch_size]
        Returns:
            Predicted noise [batch_size, data_dim]
        """
        t_emb = self.time_embed(t.float())
        h = self.input_proj(x)
        for block in self.res_blocks:
            h = block(h, t_emb)
        return self.output_proj(h)


# --- DDPM Diffusion Model ---
class DDPM:
    """Complete DDPM implementation with training and sampling."""
    def __init__(self, model: nn.Module, T: int = 1000, schedule: str = 'linear'):
        self.model = model
        self.T = T
        
        # Compute schedule
        if schedule == 'linear':
            betas = linear_beta_schedule(T)
        elif schedule == 'cosine':
            betas = cosine_beta_schedule(T)
        else:
            raise ValueError(f"Unknown schedule: {schedule}")
        
        self.params = compute_schedule_params(betas)
        # Move all params to same device as model
        self.device = next(model.parameters()).device
        for k, v in self.params.items():
            self.params[k] = v.to(self.device)
    
    def q_sample(self, x_0: torch.Tensor, t: torch.Tensor, noise: Optional[torch.Tensor] = None) -> torch.Tensor:
        """Forward process: sample x_t given x_0 and t (closed-form)."""
        if noise is None:
            noise = torch.randn_like(x_0)
        
        sqrt_alpha_bar = self.params['sqrt_alphas_cumprod'][t][:, None]
        sqrt_one_minus_alpha_bar = self.params['sqrt_one_minus_alphas_cumprod'][t][:, None]
        
        return sqrt_alpha_bar * x_0 + sqrt_one_minus_alpha_bar * noise
    
    def compute_loss(self, x_0: torch.Tensor) -> torch.Tensor:
        """Compute the simplified DDPM training loss."""
        batch_size = x_0.shape[0]
        # Sample random timesteps
        t = torch.randint(0, self.T, (batch_size,), device=self.device)
        # Sample noise
        noise = torch.randn_like(x_0)
        # Create noisy samples
        x_t = self.q_sample(x_0, t, noise)
        # Predict noise
        noise_pred = self.model(x_t, t)
        # Simple MSE loss
        return F.mse_loss(noise_pred, noise)
    
    @torch.no_grad()
    def p_sample(self, x_t: torch.Tensor, t: int) -> torch.Tensor:
        """Reverse process: single denoising step."""
        t_tensor = torch.full((x_t.shape[0],), t, device=self.device, dtype=torch.long)
        
        # Predict noise
        eps_pred = self.model(x_t, t_tensor)
        
        # Compute mean
        beta_t = self.params['betas'][t]
        sqrt_recip_alpha = self.params['sqrt_recip_alphas'][t]
        sqrt_one_minus_alpha_bar = self.params['sqrt_one_minus_alphas_cumprod'][t]
        
        mean = sqrt_recip_alpha * (x_t - (beta_t / sqrt_one_minus_alpha_bar) * eps_pred)
        
        # Add noise (except at t=0)
        if t > 0:
            noise = torch.randn_like(x_t)
            sigma = torch.sqrt(self.params['posterior_variance'][t])
            return mean + sigma * noise
        return mean
    
    @torch.no_grad()
    def sample(self, n_samples: int, data_dim: int = 2) -> torch.Tensor:
        """Generate samples by running the full reverse process."""
        self.model.eval()
        x = torch.randn(n_samples, data_dim, device=self.device)
        
        for t in reversed(range(self.T)):
            x = self.p_sample(x, t)
        
        return x

print("DDPM implementation ready!")
print(f"Model parameters: {sum(p.numel() for p in NoisePredictor().parameters()):,}")

In [0]:
# =============================================================================
# Train DDPM on 2D Swiss Roll Data
# Industrial Context: This same training loop (scaled up) powers image generators
# =============================================================================

# Initialize model and optimizer
model = NoisePredictor(data_dim=2, hidden_dim=256, time_dim=128).to(device)
diffusion = DDPM(model, T=1000, schedule='linear')
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)

# Create DataLoader
dataset = TensorDataset(data_2d.to(device))
dataloader = DataLoader(dataset, batch_size=256, shuffle=True)

# Training loop
losses = []
n_epochs = 150

print("Training DDPM on Swiss Roll data...")
for epoch in range(n_epochs):
    epoch_loss = 0.0
    for batch in dataloader:
        x_0 = batch[0]
        loss = diffusion.compute_loss(x_0)
        
        optimizer.zero_grad()
        loss.backward()
        # Gradient clipping (standard practice in diffusion models)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 30 == 0:
        print(f"  Epoch {epoch+1}/{n_epochs} | Loss: {avg_loss:.6f}")

print("\nTraining complete!")

# Generate samples
print("Generating samples (running 1000 denoising steps)...")
samples = diffusion.sample(n_samples=5000, data_dim=2).cpu()

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training loss
axes[0].plot(losses, color='darkblue', linewidth=1.5)
axes[0].set_title('Training Loss (Simplified L2)', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

# Original data
axes[1].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=2, alpha=0.5, c='steelblue')
axes[1].set_title('Original Swiss Roll Data', fontsize=12)
axes[1].set_xlim(-3.5, 3.5)
axes[1].set_ylim(-3.5, 3.5)
axes[1].set_aspect('equal')

# Generated samples
axes[2].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=2, alpha=0.5, c='crimson')
axes[2].set_title('DDPM Generated Samples', fontsize=12)
axes[2].set_xlim(-3.5, 3.5)
axes[2].set_ylim(-3.5, 3.5)
axes[2].set_aspect('equal')

plt.suptitle('DDPM Results: Learning to Generate Swiss Roll from Gaussian Noise', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Score-Based Generative Models (SGM)

**Paper**: Song & Ermon (2019) — "Generative Modeling by Estimating Gradients of the Data Distribution"

**Industrial Example**: Score-based models form the theoretical foundation of **AudioLDM** (text-to-audio synthesis by Stability AI) and are used in **molecular conformation generation** for drug discovery at companies like Schrödinger and Generate:Biomedicines.

---

### 4.1 Core Idea

Instead of learning the data distribution $$p(x)$$ directly, we learn its **score function**:

$$s_\theta(x) \approx \nabla_x \log p(x)$$

The score function tells us the direction to move to increase the probability. We can then generate samples using Langevin dynamics.

### 4.2 The Problem: Score Estimation in Low-Density Regions

Naive score matching fails in low-density regions because:
* Few training samples exist there → poor gradient estimates
* Score function is undefined where $$p(x) = 0$$

**Solution**: Perturb data with multiple levels of Gaussian noise $$\{\sigma_i\}_{i=1}^L$$ where $$\sigma_1 > \sigma_2 > \ldots > \sigma_L$$.

The perturbed distribution $$q_\sigma(x) = \int p(y) \mathcal{N}(x; y, \sigma^2 I) dy$$ fills in low-density regions.

### 4.3 Score Matching Objective

**Explicit Score Matching** (intractable in practice):
$$\mathcal{J}(\theta) = \frac{1}{2} \mathbb{E}_{p(x)}\left[\| s_\theta(x) - \nabla_x \log p(x) \|^2\right]$$

**Denoising Score Matching** (tractable, equivalent up to constants):
$$\mathcal{J}_{DSM}(\theta) = \frac{1}{2} \mathbb{E}_{q_\sigma(\tilde{x}|x) p(x)}\left[\| s_\theta(\tilde{x}, \sigma) - \nabla_{\tilde{x}} \log q_\sigma(\tilde{x}|x) \|^2\right]$$

Since $$q_\sigma(\tilde{x}|x) = \mathcal{N}(\tilde{x}; x, \sigma^2 I)$$, the target score is:

$$\nabla_{\tilde{x}} \log q_\sigma(\tilde{x}|x) = -\frac{\tilde{x} - x}{\sigma^2} = -\frac{\epsilon}{\sigma}$$

This reveals the **deep connection to DDPM**: predicting the score is equivalent to predicting the noise!

$$s_\theta(x_t, t) = -\frac{\epsilon_\theta(x_t, t)}{\sqrt{1 - \bar{\alpha}_t}}$$

### 4.4 Noise Conditional Score Network (NCSN)

Train a single network $$s_\theta(x, \sigma)$$ to estimate scores at all noise levels:

$$\mathcal{L}_{NCSN} = \sum_{i=1}^{L} \lambda(\sigma_i) \mathbb{E}_{p(x)} \mathbb{E}_{\tilde{x} \sim \mathcal{N}(x, \sigma_i^2 I)} \left[\left\| s_\theta(\tilde{x}, \sigma_i) + \frac{\tilde{x} - x}{\sigma_i^2} \right\|^2\right]$$

with $$\lambda(\sigma_i) = \sigma_i^2$$ (so each noise level contributes equally).

### 4.5 Annealed Langevin Dynamics Sampling

```
Initialize x ~ N(0, sigma_1^2 * I)
for i = 1, 2, ..., L:
    alpha_i = epsilon * (sigma_i / sigma_L)^2    # Step size
    for k = 1, 2, ..., K:
        z ~ N(0, I)
        x = x + (alpha_i / 2) * s_theta(x, sigma_i) + sqrt(alpha_i) * z
return x
```

In [0]:
# =============================================================================
# Score-Based Generative Modeling: Score Estimation + Langevin Dynamics
# Industrial Application: Foundation for molecular conformation generation
# =============================================================================

class ScoreNetwork(nn.Module):
    """Noise Conditional Score Network (NCSN).
    Estimates the score function s(x, sigma) = nabla_x log p_sigma(x)
    at multiple noise levels."""
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, n_sigmas: int = 10):
        super().__init__()
        self.n_sigmas = n_sigmas
        # Geometric sequence of noise levels
        self.sigmas = nn.Parameter(
            torch.logspace(start=math.log10(10.0), end=math.log10(0.01), steps=n_sigmas),
            requires_grad=False
        )
        
        # Sigma embedding (learned)
        self.sigma_embed = nn.Embedding(n_sigmas, hidden_dim)
        
        self.net = nn.Sequential(
            nn.Linear(data_dim + hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, x: torch.Tensor, sigma_idx: torch.Tensor) -> torch.Tensor:
        """Predict score at noise level sigma_idx."""
        sigma_emb = self.sigma_embed(sigma_idx)
        h = torch.cat([x, sigma_emb], dim=-1)
        # Output is score * sigma (for numerical stability)
        score = self.net(h)
        return score


def train_score_model(model, data, n_epochs=200, lr=1e-3):
    """Train NCSN using Denoising Score Matching."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(n_epochs):
        # Random noise level
        sigma_idx = torch.randint(0, model.n_sigmas, (data.shape[0],), device=data.device)
        sigmas = model.sigmas[sigma_idx][:, None]  # [B, 1]
        
        # Perturb data
        noise = torch.randn_like(data)
        x_perturbed = data + sigmas * noise
        
        # Target score: -noise/sigma (from denoising score matching)
        target_score = -noise / sigmas
        
        # Predict score
        predicted_score = model(x_perturbed, sigma_idx)
        
        # Weighted loss: lambda(sigma) = sigma^2
        loss = (sigmas**2 * (predicted_score - target_score)**2).mean()
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs} | Loss: {loss.item():.6f}")
    
    return losses


def annealed_langevin_dynamics(model, n_samples: int = 1000, n_steps_per_sigma: int = 100, 
                                epsilon: float = 0.01) -> torch.Tensor:
    """Sample using Annealed Langevin Dynamics.
    Starts from high noise and progressively reduces it."""
    device = model.sigmas.device
    x = torch.randn(n_samples, 2, device=device) * model.sigmas[0]
    
    trajectory = [x.clone()]
    
    for i in range(model.n_sigmas):
        sigma_i = model.sigmas[i]
        # Step size scaled by noise level
        alpha_i = epsilon * (sigma_i / model.sigmas[-1]) ** 2
        sigma_idx = torch.full((n_samples,), i, device=device, dtype=torch.long)
        
        for k in range(n_steps_per_sigma):
            z = torch.randn_like(x)
            score = model(x, sigma_idx)
            x = x + (alpha_i / 2) * score + torch.sqrt(alpha_i) * z
        
        trajectory.append(x.clone())
    
    return x, trajectory


# Train the score model
print("Training Score-Based Model (NCSN)...")
score_model = ScoreNetwork(data_dim=2, hidden_dim=256, n_sigmas=10).to(device)
score_losses = train_score_model(score_model, data_2d.to(device), n_epochs=200)

# Sample using Annealed Langevin Dynamics
print("\nSampling with Annealed Langevin Dynamics...")
score_samples, trajectory = annealed_langevin_dynamics(score_model, n_samples=3000, n_steps_per_sigma=100)
score_samples = score_samples.cpu().detach()

# Visualize
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].plot(score_losses, color='darkgreen', linewidth=1.5)
axes[0].set_title('Score Matching Loss', fontsize=12)
axes[0].set_xlabel('Epoch')
axes[0].set_yscale('log')
axes[0].grid(True, alpha=0.3)

axes[1].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=2, alpha=0.5, c='steelblue')
axes[1].set_title('Original Data', fontsize=12)
axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3.5, 3.5)
axes[1].set_aspect('equal')

axes[2].scatter(score_samples[:, 0].numpy(), score_samples[:, 1].numpy(), s=2, alpha=0.5, c='darkgreen')
axes[2].set_title('Score Model Samples\n(Annealed Langevin)', fontsize=12)
axes[2].set_xlim(-3.5, 3.5); axes[2].set_ylim(-3.5, 3.5)
axes[2].set_aspect('equal')

# Show trajectory at different sigma levels
for i, traj in enumerate(trajectory[:6]):
    traj_np = traj[:500].cpu().detach().numpy()
    alpha = 0.2 + 0.15 * i
    axes[3].scatter(traj_np[:, 0], traj_np[:, 1], s=1, alpha=alpha, 
                   label=f'σ={score_model.sigmas[i].item():.2f}' if i < len(trajectory)-1 else 'Final')
axes[3].set_title('Sampling Trajectory\n(Progressive Denoising)', fontsize=12)
axes[3].set_xlim(-4, 4); axes[3].set_ylim(-4, 4)
axes[3].legend(fontsize=8, markerscale=5)
axes[3].set_aspect('equal')

plt.suptitle('Score-Based Generative Model with Annealed Langevin Dynamics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Stochastic Differential Equations (Score SDE)

**Paper**: Song, Sohl-Dickstein, Kingma et al. (2021) — "Score-Based Generative Modeling through Stochastic Differential Equations"

**Industrial Example**: The SDE framework unifies DDPM and NCSN, and is used in **Stability AI's research** for controllable generation and in **NVIDIA's GET3D** for 3D mesh generation from images.

---

### 5.1 Continuous-Time Formulation

The discrete forward process becomes a continuous-time stochastic differential equation (SDE):

$$dx = f(x, t) \, dt + g(t) \, dw$$

where:
* $$f(x, t)$$ is the **drift coefficient** (deterministic direction)
* $$g(t)$$ is the **diffusion coefficient** (noise intensity)
* $$w$$ is a standard Wiener process (Brownian motion)

The remarkable result (Anderson, 1982): the **reverse-time SDE** exists and is:

$$dx = \left[f(x,t) - g(t)^2 \nabla_x \log p_t(x)\right] dt + g(t) \, d\bar{w}$$

where $$\bar{w}$$ is a reverse-time Wiener process and $$\nabla_x \log p_t(x)$$ is the score at time $$t$$.

### 5.2 Specific SDE Types

#### Variance Preserving SDE (VP-SDE) — Continuous DDPM

$$dx = -\frac{1}{2}\beta(t) x \, dt + \sqrt{\beta(t)} \, dw$$

* Drift $$f(x,t) = -\frac{1}{2}\beta(t) x$$ shrinks the signal
* Diffusion $$g(t) = \sqrt{\beta(t)}$$ adds noise
* The marginal variance is preserved near 1 at all times
* Setting $$\beta(t) = \beta_{\min} + t(\beta_{\max} - \beta_{\min})$$ recovers DDPM

#### Variance Exploding SDE (VE-SDE) — Continuous NCSN

$$dx = \sqrt{\frac{d[\sigma^2(t)]}{dt}} \, dw$$

* Zero drift: signal is not scaled
* Variance grows over time ("explodes")
* Setting $$\sigma(t) = \sigma_{\min}(\sigma_{\max}/\sigma_{\min})^t$$ recovers NCSN

### 5.3 Probability Flow ODE

For every SDE, there exists a deterministic ODE with the **same marginals** $$p_t(x)$$:

$$dx = \left[f(x,t) - \frac{1}{2}g(t)^2 \nabla_x \log p_t(x)\right] dt$$

This is called the **Probability Flow ODE**. Its advantages:
* Deterministic: same initial noise → same output (useful for interpolation)
* Enables exact likelihood computation via the change-of-variables formula
* Can use adaptive ODE solvers for faster sampling

### 5.4 Unified View

| Model | SDE Type | Drift $$f(x,t)$$ | Diffusion $$g(t)$$ |
| --- | --- | --- | --- |
| DDPM | VP-SDE | $$-\frac{1}{2}\beta(t)x$$ | $$\sqrt{\beta(t)}$$ |
| NCSN/SMLD | VE-SDE | $$0$$ | $$\sigma(t)\sqrt{2\log(\sigma_{\max}/\sigma_{\min})}$$ |
| Sub-VP SDE | Sub-VP | $$-\frac{1}{2}\beta(t)x$$ | $$\sqrt{\beta(t)(1 - e^{-2\int_0^t \beta(s)ds})}$$ |

In [0]:
# =============================================================================
# Score SDE: Unified Framework (VP-SDE and VE-SDE)
# Industrial Application: NVIDIA's GET3D for 3D mesh generation
# =============================================================================

class VPSDE:
    """Variance Preserving SDE (continuous-time DDPM).
    dx = -0.5 * beta(t) * x * dt + sqrt(beta(t)) * dw
    """
    def __init__(self, beta_min: float = 0.1, beta_max: float = 20.0):
        self.beta_min = beta_min
        self.beta_max = beta_max
    
    def beta(self, t: torch.Tensor) -> torch.Tensor:
        """Linear schedule in continuous time."""
        return self.beta_min + t * (self.beta_max - self.beta_min)
    
    def marginal_params(self, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Compute mean coefficient and std of q(x_t | x_0)."""
        log_mean_coeff = -0.25 * t**2 * (self.beta_max - self.beta_min) - 0.5 * t * self.beta_min
        mean_coeff = torch.exp(log_mean_coeff)
        std = torch.sqrt(1.0 - torch.exp(2.0 * log_mean_coeff))
        return mean_coeff, std
    
    def perturb(self, x_0: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample x_t given x_0 and t."""
        mean_coeff, std = self.marginal_params(t)
        noise = torch.randn_like(x_0)
        x_t = mean_coeff[:, None] * x_0 + std[:, None] * noise
        return x_t, noise
    
    def drift_and_diffusion(self, x: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Return f(x,t) and g(t) for the SDE."""
        beta_t = self.beta(t)
        drift = -0.5 * beta_t[:, None] * x
        diffusion = torch.sqrt(beta_t)
        return drift, diffusion


class VESDE:
    """Variance Exploding SDE (continuous-time NCSN).
    dx = sqrt(d[sigma^2(t)]/dt) * dw
    """
    def __init__(self, sigma_min: float = 0.01, sigma_max: float = 50.0):
        self.sigma_min = sigma_min
        self.sigma_max = sigma_max
    
    def sigma(self, t: torch.Tensor) -> torch.Tensor:
        """Geometric schedule for noise levels."""
        return self.sigma_min * (self.sigma_max / self.sigma_min) ** t
    
    def marginal_params(self, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Mean is 1 (no scaling), std is sigma(t)."""
        return torch.ones_like(t), self.sigma(t)
    
    def perturb(self, x_0: torch.Tensor, t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Sample x_t given x_0 and t."""
        sigma_t = self.sigma(t)
        noise = torch.randn_like(x_0)
        x_t = x_0 + sigma_t[:, None] * noise
        return x_t, noise


# --- Euler-Maruyama SDE Solver (for reverse-time SDE) ---
def euler_maruyama_reverse(score_fn, sde, n_samples: int, data_dim: int, 
                          n_steps: int = 500, device='cpu') -> torch.Tensor:
    """Solve reverse-time SDE using Euler-Maruyama discretization.
    dx = [f(x,t) - g(t)^2 * score(x,t)] dt + g(t) dw_bar
    """
    dt = -1.0 / n_steps  # Negative because we go from t=1 to t=0
    
    # Start from prior (Gaussian)
    _, std_final = sde.marginal_params(torch.ones(1, device=device))
    x = torch.randn(n_samples, data_dim, device=device) * std_final.item()
    
    times = torch.linspace(1.0, 1e-4, n_steps, device=device)
    
    for t_val in times:
        t = torch.full((n_samples,), t_val, device=device)
        
        # Get drift and diffusion
        drift, diffusion = sde.drift_and_diffusion(x, t)
        g_squared = diffusion[:, None] ** 2
        
        # Get score
        score = score_fn(x, t)
        
        # Reverse drift: f(x,t) - g(t)^2 * score(x,t)
        reverse_drift = drift - g_squared * score
        
        # Euler-Maruyama step (note: dt is negative)
        noise = torch.randn_like(x)
        x = x + reverse_drift * abs(dt) + diffusion[:, None] * math.sqrt(abs(dt)) * noise
    
    return x


# --- Probability Flow ODE Solver ---
def probability_flow_ode(score_fn, sde, n_samples: int, data_dim: int,
                         n_steps: int = 500, device='cpu') -> torch.Tensor:
    """Solve Probability Flow ODE (deterministic).
    dx = [f(x,t) - 0.5 * g(t)^2 * score(x,t)] dt
    """
    dt = -1.0 / n_steps
    
    _, std_final = sde.marginal_params(torch.ones(1, device=device))
    x = torch.randn(n_samples, data_dim, device=device) * std_final.item()
    
    times = torch.linspace(1.0, 1e-4, n_steps, device=device)
    
    for t_val in times:
        t = torch.full((n_samples,), t_val, device=device)
        drift, diffusion = sde.drift_and_diffusion(x, t)
        g_squared = diffusion[:, None] ** 2
        score = score_fn(x, t)
        
        # ODE: no noise term
        ode_drift = drift - 0.5 * g_squared * score
        x = x + ode_drift * abs(dt)
    
    return x


# --- Train a score model for VP-SDE ---
class ContinuousScoreNet(nn.Module):
    """Score network for continuous time t in [0, 1]."""
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, time_dim: int = 64):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(t * 1000)  # Scale for embedding
        h = torch.cat([x, t_emb], dim=-1)
        return self.net(h)


# Train
print("Training continuous-time score model (VP-SDE)...")
vp_sde = VPSDE(beta_min=0.1, beta_max=20.0)
score_net = ContinuousScoreNet(data_dim=2).to(device)
optimizer = torch.optim.Adam(score_net.parameters(), lr=2e-3)

data_device = data_2d.to(device)
sde_losses = []

for epoch in range(300):
    # Random continuous time
    t = torch.rand(data_device.shape[0], device=device) * (1.0 - 1e-4) + 1e-4
    
    # Perturb data
    x_t, noise = vp_sde.perturb(data_device, t)
    _, std = vp_sde.marginal_params(t)
    
    # Target: -noise/std (the true score)
    target = -noise / std[:, None]
    
    # Predict score
    score_pred = score_net(x_t, t)
    
    # Loss weighted by std^2
    loss = ((std[:, None]**2) * (score_pred - target)**2).mean()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    sde_losses.append(loss.item())
    
    if (epoch + 1) % 100 == 0:
        print(f"  Epoch {epoch+1}/300 | Loss: {loss.item():.4f}")

# Sample using both SDE and ODE
print("\nSampling with Reverse SDE (stochastic)...")
samples_sde = euler_maruyama_reverse(
    score_net, vp_sde, n_samples=2000, data_dim=2, n_steps=500, device=device
).cpu().detach()

print("Sampling with Probability Flow ODE (deterministic)...")
samples_ode = probability_flow_ode(
    score_net, vp_sde, n_samples=2000, data_dim=2, n_steps=500, device=device
).cpu().detach()

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=2, alpha=0.4, c='steelblue')
axes[0].set_title('Original Data', fontsize=12)
axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5); axes[0].set_aspect('equal')

axes[1].scatter(samples_sde[:, 0].numpy(), samples_sde[:, 1].numpy(), s=2, alpha=0.4, c='purple')
axes[1].set_title('Reverse SDE Samples\n(Stochastic)', fontsize=12)
axes[1].set_xlim(-3.5, 3.5); axes[1].set_ylim(-3.5, 3.5); axes[1].set_aspect('equal')

axes[2].scatter(samples_ode[:, 0].numpy(), samples_ode[:, 1].numpy(), s=2, alpha=0.4, c='darkorange')
axes[2].set_title('Probability Flow ODE Samples\n(Deterministic)', fontsize=12)
axes[2].set_xlim(-3.5, 3.5); axes[2].set_ylim(-3.5, 3.5); axes[2].set_aspect('equal')

plt.suptitle('Score SDE: Stochastic vs Deterministic Sampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Denoising Diffusion Implicit Models (DDIM)

**Paper**: Song, Meng, Ermon (2021) — "Denoising Diffusion Implicit Models"

**Industrial Example**: DDIM's accelerated sampling enables **real-time image editing** in products like **Adobe Firefly** (image inpainting with 20-50 steps instead of 1000), **Midjourney** (fast inference for interactive creation), and **Apple's Core ML Stable Diffusion** (on-device generation on iPhones).

---

### 6.1 Motivation

DDPM requires $$T = 1000$$ sequential denoising steps at generation time, making it slow. DDIM addresses this by:
1. Defining a **non-Markovian** forward process with the same marginals
2. Enabling **deterministic** generation (no stochasticity in the reverse process)
3. Allowing **sub-sequence sampling** with drastically fewer steps (e.g., 50 instead of 1000)

### 6.2 Non-Markovian Forward Process

DDIM defines a family of forward processes indexed by $$\sigma$$ that all share the same marginals $$q(x_t|x_0)$$:

$$q_\sigma(x_{t-1} | x_t, x_0) = \mathcal{N}\left(\sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \frac{x_t - \sqrt{\bar{\alpha}_t} x_0}{\sqrt{1 - \bar{\alpha}_t}}, \; \sigma_t^2 I\right)$$

where $$\sigma_t$$ controls the stochasticity:
* $$\sigma_t = \sqrt{\frac{(1-\bar{\alpha}_{t-1})}{(1-\bar{\alpha}_t)}} \cdot \sqrt{1 - \frac{\bar{\alpha}_t}{\bar{\alpha}_{t-1}}}$$ recovers **DDPM**
* $$\sigma_t = 0$$ gives **DDIM** (fully deterministic)

### 6.3 DDIM Sampling Update Rule

Given a trained $$\epsilon_\theta$$ (from any DDPM), the DDIM update is:

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \underbrace{\left(\frac{x_t - \sqrt{1-\bar{\alpha}_t} \cdot \epsilon_\theta(x_t, t)}{\sqrt{\bar{\alpha}_t}}\right)}_{\text{"predicted } x_0\text{"}} + \underbrace{\sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \cdot \epsilon_\theta(x_t, t)}_{\text{"direction pointing to } x_t\text{"}} + \underbrace{\sigma_t \epsilon_t}_{\text{noise}}$$

When $$\sigma_t = 0$$ (deterministic DDIM):

$$x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} \cdot \hat{x}_0(x_t) + \sqrt{1 - \bar{\alpha}_{t-1}} \cdot \epsilon_\theta(x_t, t)$$

### 6.4 Sub-Sequence Sampling

Instead of using all $$T$$ timesteps, sample a sub-sequence $$\tau = [\tau_1, \tau_2, \ldots, \tau_S]$$ where $$S \ll T$$:

* Example: $$T=1000, S=50 \Rightarrow \tau = [0, 20, 40, \ldots, 980, 999]$$
* The same pre-trained model works — no retraining needed!
* Quality degrades gracefully: 50 steps ≈ 90% of 1000-step quality

### 6.5 Latent Space Properties

Because DDIM is deterministic, the initial noise $$x_T$$ uniquely determines the output $$x_0$$:
* **Semantic interpolation**: Interpolate between two $$x_T$$ vectors to get meaningful transitions
* **Inversion**: Given a real image $$x_0$$, find its latent $$x_T$$ by running the forward ODE
* **Editing**: Modify the latent and re-denoise for semantically meaningful edits

In [0]:
# =============================================================================
# DDIM: Accelerated Deterministic Sampling
# Industrial Context: Enables real-time generation in Adobe Firefly, Midjourney
# =============================================================================

class DDIMSampler:
    """DDIM sampling from a pre-trained DDPM model.
    Can use any subset of timesteps for accelerated generation."""
    
    def __init__(self, model: nn.Module, params: dict, T: int = 1000):
        self.model = model
        self.params = params
        self.T = T
    
    @torch.no_grad()
    def sample(self, n_samples: int, data_dim: int, n_steps: int = 50, 
               eta: float = 0.0, device='cpu') -> Tuple[torch.Tensor, list]:
        """DDIM sampling with configurable number of steps and stochasticity.
        
        Args:
            n_samples: Number of samples to generate
            data_dim: Dimensionality of data
            n_steps: Number of denoising steps (can be much less than T)
            eta: Stochasticity parameter (0=DDIM deterministic, 1=DDPM)
        """
        self.model.eval()
        
        # Create sub-sequence of timesteps
        # E.g., if T=1000 and n_steps=50: [999, 979, 959, ..., 19]
        step_size = self.T // n_steps
        timesteps = list(range(0, self.T, step_size))
        timesteps = list(reversed(timesteps))
        
        # Start from pure noise
        x = torch.randn(n_samples, data_dim, device=device)
        trajectory = [x.clone().cpu()]
        
        alphas_cumprod = self.params['alphas_cumprod'].to(device)
        
        for i in range(len(timesteps) - 1):
            t_curr = timesteps[i]
            t_prev = timesteps[i + 1]
            
            t_tensor = torch.full((n_samples,), t_curr, device=device, dtype=torch.long)
            
            # Predict noise
            eps_pred = self.model(x, t_tensor)
            
            # Get alpha_bar values
            alpha_bar_t = alphas_cumprod[t_curr]
            alpha_bar_t_prev = alphas_cumprod[t_prev]
            
            # Predict x_0
            x0_pred = (x - torch.sqrt(1 - alpha_bar_t) * eps_pred) / torch.sqrt(alpha_bar_t)
            
            # Compute sigma (stochasticity)
            sigma_t = eta * torch.sqrt(
                (1 - alpha_bar_t_prev) / (1 - alpha_bar_t) * (1 - alpha_bar_t / alpha_bar_t_prev)
            )
            
            # Direction pointing to x_t
            dir_xt = torch.sqrt(1 - alpha_bar_t_prev - sigma_t**2) * eps_pred
            
            # DDIM update
            x = torch.sqrt(alpha_bar_t_prev) * x0_pred + dir_xt
            
            # Add noise if eta > 0
            if eta > 0 and t_prev > 0:
                noise = torch.randn_like(x)
                x = x + sigma_t * noise
            
            trajectory.append(x.clone().cpu())
        
        return x, trajectory


# Use the DDPM model we trained earlier
ddim_sampler = DDIMSampler(model, params, T=1000)

# Compare different step counts
print("Comparing DDPM (1000 steps) vs DDIM (various steps)...")
step_configs = [
    (1000, 0.0, 'DDIM (1000 steps, η=0)'),
    (200, 0.0, 'DDIM (200 steps, η=0)'),
    (50, 0.0, 'DDIM (50 steps, η=0)'),
    (20, 0.0, 'DDIM (20 steps, η=0)'),
    (50, 1.0, 'DDIM (50 steps, η=1 ≈ DDPM)'),
]

fig, axes = plt.subplots(1, len(step_configs) + 1, figsize=(24, 4))

# Original data
axes[0].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=2, alpha=0.4, c='steelblue')
axes[0].set_title('Original Data', fontsize=11)
axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5); axes[0].set_aspect('equal')

for idx, (steps, eta, label) in enumerate(step_configs):
    samples_ddim, _ = ddim_sampler.sample(
        n_samples=2000, data_dim=2, n_steps=steps, eta=eta, device=device
    )
    samples_ddim = samples_ddim.cpu()
    
    axes[idx+1].scatter(samples_ddim[:, 0].numpy(), samples_ddim[:, 1].numpy(), 
                       s=2, alpha=0.4, c='teal')
    axes[idx+1].set_title(label, fontsize=10)
    axes[idx+1].set_xlim(-3.5, 3.5); axes[idx+1].set_ylim(-3.5, 3.5)
    axes[idx+1].set_aspect('equal')

plt.suptitle('DDIM: Trading Steps for Speed (Same Pre-trained Model)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Demonstrate deterministic interpolation (DDIM latent space)
print("\nDemonstrating DDIM Latent Space Interpolation...")
torch.manual_seed(123)
z1 = torch.randn(1, 2, device=device)
z2 = torch.randn(1, 2, device=device)

fig, axes = plt.subplots(1, 7, figsize=(21, 3))
for idx, alpha in enumerate(np.linspace(0, 1, 7)):
    # Spherical interpolation (slerp)
    omega = torch.acos(torch.clamp(F.cosine_similarity(z1, z2), -1, 1))
    z_interp = (torch.sin((1-alpha)*omega)/torch.sin(omega)) * z1 + (torch.sin(alpha*omega)/torch.sin(omega)) * z2
    
    # Expand and denoise
    z_batch = z_interp.expand(500, -1)
    samples_interp, _ = ddim_sampler.sample(n_samples=500, data_dim=2, n_steps=100, eta=0.0, device=device)
    samples_interp = samples_interp.cpu()
    
    axes[idx].scatter(samples_interp[:, 0].numpy(), samples_interp[:, 1].numpy(), s=3, alpha=0.5, c='coral')
    axes[idx].set_title(f'α = {alpha:.2f}', fontsize=10)
    axes[idx].set_xlim(-3.5, 3.5); axes[idx].set_ylim(-3.5, 3.5)
    axes[idx].set_aspect('equal')

plt.suptitle('DDIM: Deterministic Nature Enables Consistent Sampling', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Latent Diffusion Models (LDM)

**Paper**: Rombach, Blattmann et al. (2022) — "High-Resolution Image Synthesis with Latent Diffusion Models"

**Industrial Example**: LDMs are the architecture behind **Stable Diffusion** (Stability AI), **DALL·E 3** (OpenAI), **Midjourney v5+**, **Adobe Firefly**, and **Runway Gen-2** (video). By working in a compressed latent space, they reduce compute by $$\sim 50\times$$ compared to pixel-space diffusion.

---

### 7.1 Motivation: The Computational Bottleneck

Pixel-space diffusion on high-resolution images (e.g., $$512 \times 512 \times 3$$) is extremely expensive:
* Each forward/backward pass processes 786,432 dimensions
* The UNet must capture both low-level (texture) and high-level (semantics) features
* Training requires hundreds of GPU-days on A100s

**Key Insight**: Most of the perceptual information in images can be encoded in a much smaller latent space. High-frequency, imperceptible details are captured by the decoder.

### 7.2 Architecture: Two-Stage Approach

```
Stage 1: Perceptual Compression (Autoencoder)
┌───────────┐    ┌─────────────┐    ┌───────────┐
│  Image x  │ →→ │  Encoder E  │ →→ │  Latent z  │
│ 512x512x3 │    │             │    │  64x64x4   │
└───────────┘    └─────────────┘    └───────────┘
                                          │
                                          ↓ (64x compression)
Stage 2: Latent Diffusion
┌───────────┐    ┌─────────────┐    ┌───────────┐
│  Noise z_T│ →→ │  UNet ε_θ   │ →→ │ Denoised z │
│  64x64x4  │    │ + Attention │    │  64x64x4   │
└───────────┘    └─────────────┘    └───────────┘
                      ↑                       │
               Condition c              ┌─────┴─────┐
               (text, class)            │ Decoder D │ →→ Image
                                        └───────────┘
```

### 7.3 Stage 1: Autoencoder Training

The autoencoder learns a perceptual compression:

$$\mathcal{L}_{AE} = \underbrace{\| x - D(E(x)) \|^2}_{\text{reconstruction}} + \underbrace{\lambda_{\text{perc}} \mathcal{L}_{\text{LPIPS}}}_{\text{perceptual}} + \underbrace{\lambda_{\text{adv}} \mathcal{L}_{\text{GAN}}}_{\text{adversarial}} + \underbrace{\lambda_{\text{reg}} \mathcal{L}_{\text{reg}}}_{\text{KL or VQ}}$$

Regularization options:
* **KL-regularization**: Penalize $$D_{KL}(q(z|x) \| \mathcal{N}(0, I))$$ (like VAE)
* **VQ-regularization**: Use Vector Quantization (like VQ-VAE)

The compression ratio $$f = H/h$$ (typically $$f = 8$$: $$512 \to 64$$) determines the tradeoff between perceptual quality and efficiency.

### 7.4 Stage 2: Latent Diffusion

Run standard diffusion in the latent space:

$$\mathcal{L}_{LDM} = \mathbb{E}_{z_0 = E(x), t, \epsilon}\left[\| \epsilon - \epsilon_\theta(z_t, t, c) \|^2\right]$$

where $$z_t = \sqrt{\bar{\alpha}_t} z_0 + \sqrt{1-\bar{\alpha}_t} \epsilon$$ and $$c$$ is the conditioning signal.

### 7.5 Cross-Attention Conditioning

Conditioning (text, class, layout) is injected via cross-attention in the UNet:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{Q K^T}{\sqrt{d}}\right) V$$

where:
* $$Q = W_Q \cdot \phi(z_t)$$ — queries from the noisy latent
* $$K = W_K \cdot \tau_\theta(c)$$ — keys from the condition encoder
* $$V = W_V \cdot \tau_\theta(c)$$ — values from the condition encoder

For text conditioning: $$\tau_\theta$$ is typically a frozen CLIP or T5 text encoder.

In [0]:
# =============================================================================
# Latent Diffusion Model: Simplified Architecture Demonstration
# Industrial Context: Core architecture of Stable Diffusion, DALL-E 3, Midjourney
# =============================================================================

# --- Simple Variational Autoencoder (Stage 1: Perceptual Compression) ---
class SimpleVAE(nn.Module):
    """A simple VAE for demonstration of the two-stage LDM approach.
    In production: This would be a convolutional KL-autoencoder with
    perceptual loss, adversarial loss, and 64x spatial compression."""
    def __init__(self, data_dim: int = 2, latent_dim: int = 2):
        super().__init__()
        # Encoder: data -> latent (mu, log_var)
        self.encoder = nn.Sequential(
            nn.Linear(data_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
        )
        self.fc_mu = nn.Linear(128, latent_dim)
        self.fc_logvar = nn.Linear(128, latent_dim)
        
        # Decoder: latent -> data
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, data_dim),
        )
    
    def encode(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        h = self.encoder(x)
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu: torch.Tensor, logvar: torch.Tensor) -> torch.Tensor:
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + std * eps
    
    def decode(self, z: torch.Tensor) -> torch.Tensor:
        return self.decoder(z)
    
    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        x_recon = self.decode(z)
        return x_recon, mu, logvar


def vae_loss(x_recon, x, mu, logvar, kl_weight=0.001):
    """VAE loss = Reconstruction + KL divergence."""
    recon_loss = F.mse_loss(x_recon, x, reduction='mean')
    kl_loss = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + kl_weight * kl_loss


# --- Train VAE (Stage 1) ---
print("="*60)
print("LATENT DIFFUSION MODEL: Two-Stage Training")
print("="*60)
print("\nStage 1: Training Autoencoder (Perceptual Compression)...")

vae = SimpleVAE(data_dim=2, latent_dim=2).to(device)
vae_optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

for epoch in range(200):
    for batch in dataloader:
        x = batch[0]
        x_recon, mu, logvar = vae(x)
        loss = vae_loss(x_recon, x, mu, logvar)
        vae_optimizer.zero_grad()
        loss.backward()
        vae_optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/200 | VAE Loss: {loss.item():.4f}")

# --- Train Diffusion in Latent Space (Stage 2) ---
print("\nStage 2: Training Diffusion Model in Latent Space...")

# Encode all data to latent space
vae.eval()
with torch.no_grad():
    mu_all, _ = vae.encode(data_2d.to(device))
    latent_data = mu_all  # Use mean encoding as latent data

print(f"  Data dimension: {data_2d.shape[1]} -> Latent dimension: {latent_data.shape[1]}")
print(f"  (In Stable Diffusion: 512x512x3 -> 64x64x4, a 48x compression!)")

# Train a diffusion model on latent representations
latent_model = NoisePredictor(data_dim=2, hidden_dim=128, time_dim=64).to(device)
latent_diffusion = DDPM(latent_model, T=500, schedule='cosine')
latent_optimizer = torch.optim.Adam(latent_model.parameters(), lr=3e-4)

latent_dataset = TensorDataset(latent_data)
latent_loader = DataLoader(latent_dataset, batch_size=256, shuffle=True)

for epoch in range(100):
    for batch in latent_loader:
        z0 = batch[0]
        loss = latent_diffusion.compute_loss(z0)
        latent_optimizer.zero_grad()
        loss.backward()
        latent_optimizer.step()
    
    if (epoch + 1) % 25 == 0:
        print(f"  Epoch {epoch+1}/100 | Latent Diffusion Loss: {loss.item():.6f}")

# --- Generate: Sample latent -> Decode to data space ---
print("\nGenerating samples: Noise -> Latent Diffusion -> VAE Decode -> Data")
latent_samples = latent_diffusion.sample(n_samples=3000, data_dim=2)

with torch.no_grad():
    generated_data = vae.decode(latent_samples).cpu()

# Visualization
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=2, alpha=0.4, c='steelblue')
axes[0].set_title('Original Data (x)', fontsize=12)
axes[0].set_xlim(-3.5, 3.5); axes[0].set_ylim(-3.5, 3.5); axes[0].set_aspect('equal')

axes[1].scatter(latent_data[:, 0].cpu().numpy(), latent_data[:, 1].cpu().numpy(), s=2, alpha=0.4, c='purple')
axes[1].set_title('Latent Space (z = E(x))', fontsize=12)
axes[1].set_aspect('equal')

axes[2].scatter(latent_samples[:, 0].cpu().numpy(), latent_samples[:, 1].cpu().numpy(), s=2, alpha=0.4, c='darkorange')
axes[2].set_title('Generated Latents\n(from Diffusion)', fontsize=12)
axes[2].set_aspect('equal')

axes[3].scatter(generated_data[:, 0].numpy(), generated_data[:, 1].numpy(), s=2, alpha=0.4, c='crimson')
axes[3].set_title('Generated Data\n(D(z_generated))', fontsize=12)
axes[3].set_xlim(-3.5, 3.5); axes[3].set_ylim(-3.5, 3.5); axes[3].set_aspect('equal')

plt.suptitle('Latent Diffusion Model: Encode → Diffuse in Latent Space → Decode', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Conditional Generation & Guidance Techniques

**Papers**: 
* Dhariwal & Nichol (2021) — "Diffusion Models Beat GANs on Image Synthesis" (Classifier Guidance)
* Ho & Salimans (2022) — "Classifier-Free Diffusion Guidance"

**Industrial Examples**:
* **Classifier Guidance**: Used in OpenAI's **GLIDE** for initial text-to-image systems
* **Classifier-Free Guidance (CFG)**: The standard technique in **Stable Diffusion**, **DALL·E 3**, **Imagen**, **Midjourney** — the $$w$$ (guidance scale) parameter users adjust for creativity vs. fidelity

---

### 8.1 The Conditioning Problem

We want to sample from $$p(x | c)$$ where $$c$$ is a condition (text, class label, image). Using Bayes' rule:

$$\nabla_x \log p(x | c) = \nabla_x \log p(x) + \nabla_x \log p(c | x)$$

The conditional score decomposes into an **unconditional score** plus a **classifier gradient**.

### 8.2 Classifier Guidance

Train a noise-aware classifier $$p_\phi(c | x_t, t)$$ on noisy data, then modify the sampling:

$$\hat{\epsilon}(x_t, t, c) = \epsilon_\theta(x_t, t) - \sqrt{1 - \bar{\alpha}_t} \cdot w \cdot \nabla_{x_t} \log p_\phi(c | x_t, t)$$

where $$w$$ is the **guidance scale** controlling the strength of conditioning.

**Pros**: Can use any pre-trained diffusion model + any classifier
**Cons**: Requires training a separate noise-robust classifier; gradient computation is expensive

### 8.3 Classifier-Free Guidance (CFG)

The breakthrough idea: **no separate classifier needed**. Instead, train a single model on both conditional and unconditional generation:

During training, randomly drop the conditioning $$c$$ (replace with null $$\emptyset$$) with probability $$p_{\text{uncond}}$$ (typically 10-20%):

$$\epsilon_\theta(x_t, t, c) \quad \text{and} \quad \epsilon_\theta(x_t, t, \emptyset)$$

During sampling, extrapolate between unconditional and conditional predictions:

$$\hat{\epsilon}(x_t, t, c) = \underbrace{\epsilon_\theta(x_t, t, \emptyset)}_{\text{unconditional}} + w \cdot \underbrace{\left(\epsilon_\theta(x_t, t, c) - \epsilon_\theta(x_t, t, \emptyset)\right)}_{\text{direction toward condition}}$$

Simplified:
$$\hat{\epsilon} = (1 - w) \cdot \epsilon_\theta(x_t, t, \emptyset) + w \cdot \epsilon_\theta(x_t, t, c)$$

**Interpretation**: The guidance scale $$w$$ controls the trade-off:
* $$w = 0$$: Pure unconditional generation (diverse but may not match prompt)
* $$w = 1$$: Standard conditional generation (balanced)
* $$w > 1$$: Amplified conditioning (higher fidelity to prompt, less diversity)
* $$w = 7.5$$: Typical setting in Stable Diffusion

### 8.4 Effect of Guidance Scale

| Guidance Scale $$w$$ | Behavior | FID↓ | CLIP Score↑ |
| --- | --- | --- | --- |
| 1.0 | No guidance (baseline) | Best | Low |
| 3.0 | Mild guidance | Good | Medium |
| 7.5 | Standard (Stable Diffusion) | Moderate | High |
| 15.0 | Strong guidance | Poor | Very High |
| 30.0+ | Over-saturated, artifacts | Bad | Peaks then falls |

The optimal $$w$$ minimizes the Pareto frontier between FID (quality/diversity) and CLIP score (text-image alignment).

In [0]:
# =============================================================================
# Classifier-Free Guidance: Conditional Diffusion Model
# Industrial Context: The 'w' parameter users adjust in Stable Diffusion/Midjourney
# =============================================================================

class ConditionalNoisePredictor(nn.Module):
    """Noise predictor with class conditioning and dropout for CFG.
    During training: condition is randomly dropped (replaced with null embedding)
    During inference: run both conditional and unconditional, interpolate."""
    
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, 
                 time_dim: int = 128, n_classes: int = 3, cond_drop_prob: float = 0.1):
        super().__init__()
        self.cond_drop_prob = cond_drop_prob
        self.n_classes = n_classes
        
        # Time embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        
        # Class embedding (class 0 = unconditional/null token)
        self.class_embed = nn.Embedding(n_classes + 1, time_dim)  # +1 for null class
        
        # Main network
        self.net = nn.Sequential(
            nn.Linear(data_dim + 2 * time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, x: torch.Tensor, t: torch.Tensor, 
                class_label: torch.Tensor, force_uncond: bool = False) -> torch.Tensor:
        t_emb = self.time_embed(t.float())
        
        if force_uncond:
            # Unconditional: use null class token
            c_emb = self.class_embed(torch.zeros_like(class_label))
        else:
            if self.training:
                # Randomly drop conditioning during training
                mask = torch.bernoulli(torch.full_like(class_label.float(), self.cond_drop_prob)).bool()
                labels = class_label.clone()
                labels[mask] = 0  # 0 = null/unconditional
                c_emb = self.class_embed(labels)
            else:
                c_emb = self.class_embed(class_label)
        
        h = torch.cat([x, t_emb, c_emb], dim=-1)
        return self.net(h)


# Generate labeled data: 3 clusters (representing 3 classes)
from sklearn.datasets import make_moons, make_circles

n_per_class = 2000
# Class 1: Moon shape
moons, _ = make_moons(n_samples=n_per_class, noise=0.05)
moons = torch.tensor(moons, dtype=torch.float32) * 1.5
# Class 2: Circle
circle, _ = make_circles(n_samples=n_per_class, noise=0.03, factor=0.5)
circle = torch.tensor(circle, dtype=torch.float32) * 2.0 + torch.tensor([3.0, 0.0])
# Class 3: Gaussian cluster
gaussian = torch.randn(n_per_class, 2) * 0.5 + torch.tensor([-2.0, 2.0])

# Combine
class_data = torch.cat([moons, circle, gaussian], dim=0)
class_labels = torch.cat([
    torch.ones(n_per_class, dtype=torch.long),   # Class 1
    torch.full((n_per_class,), 2, dtype=torch.long),  # Class 2
    torch.full((n_per_class,), 3, dtype=torch.long),  # Class 3
])

# Normalize
class_data = (class_data - class_data.mean(0)) / class_data.std(0)

# Train conditional model
print("Training Conditional Diffusion Model with CFG...")
cond_model = ConditionalNoisePredictor(
    data_dim=2, hidden_dim=256, time_dim=128, n_classes=3, cond_drop_prob=0.15
).to(device)

cond_optimizer = torch.optim.Adam(cond_model.parameters(), lr=3e-4)
cond_betas = cosine_beta_schedule(500)
cond_params = compute_schedule_params(cond_betas)
for k, v in cond_params.items():
    cond_params[k] = v.to(device)

cond_dataset = TensorDataset(class_data.to(device), class_labels.to(device))
cond_loader = DataLoader(cond_dataset, batch_size=512, shuffle=True)

T_cond = 500
for epoch in range(150):
    for x_0, labels in cond_loader:
        batch_size = x_0.shape[0]
        t = torch.randint(0, T_cond, (batch_size,), device=device)
        noise = torch.randn_like(x_0)
        
        x_t = cond_params['sqrt_alphas_cumprod'][t][:, None] * x_0 + \
              cond_params['sqrt_one_minus_alphas_cumprod'][t][:, None] * noise
        
        noise_pred = cond_model(x_t, t, labels)
        loss = F.mse_loss(noise_pred, noise)
        
        cond_optimizer.zero_grad()
        loss.backward()
        cond_optimizer.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/150 | Loss: {loss.item():.6f}")


# --- CFG Sampling ---
@torch.no_grad()
def cfg_sample(model, params, target_class: int, n_samples: int = 1000, 
               guidance_scale: float = 3.0, T: int = 500) -> torch.Tensor:
    """Sample with Classifier-Free Guidance.
    eps_hat = eps_uncond + w * (eps_cond - eps_uncond)
    """
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    labels = torch.full((n_samples,), target_class, device=device, dtype=torch.long)
    
    for t in reversed(range(T)):
        t_tensor = torch.full((n_samples,), t, device=device, dtype=torch.long)
        
        # Conditional prediction
        eps_cond = model(x, t_tensor, labels, force_uncond=False)
        # Unconditional prediction
        eps_uncond = model(x, t_tensor, labels, force_uncond=True)
        
        # CFG interpolation
        eps_guided = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        
        # Denoise step
        beta_t = params['betas'][t]
        sqrt_recip_alpha = params['sqrt_recip_alphas'][t]
        sqrt_one_minus_alpha_bar = params['sqrt_one_minus_alphas_cumprod'][t]
        
        mean = sqrt_recip_alpha * (x - (beta_t / sqrt_one_minus_alpha_bar) * eps_guided)
        
        if t > 0:
            noise = torch.randn_like(x)
            sigma = torch.sqrt(params['posterior_variance'][t])
            x = mean + sigma * noise
        else:
            x = mean
    
    return x


# Generate with different guidance scales
print("\nGenerating samples with various guidance scales...")
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 1: Effect of guidance scale for Class 1 (Moons)
guidance_scales = [0.0, 1.0, 3.0, 7.5]
for idx, w in enumerate(guidance_scales):
    samples = cfg_sample(cond_model, cond_params, target_class=1, 
                        n_samples=1500, guidance_scale=w, T=500).cpu()
    axes[0, idx].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=3, alpha=0.5, c='crimson')
    axes[0, idx].set_title(f'Class 1 (Moons) | w = {w}', fontsize=11)
    axes[0, idx].set_xlim(-4, 4); axes[0, idx].set_ylim(-4, 4)
    axes[0, idx].set_aspect('equal')

# Row 2: Different classes with w=3.0
axes[1, 0].scatter(class_data[:, 0].numpy(), class_data[:, 1].numpy(), 
                   s=2, alpha=0.3, c=class_labels.numpy(), cmap='Set1')
axes[1, 0].set_title('Training Data\n(3 classes)', fontsize=11)
axes[1, 0].set_xlim(-4, 4); axes[1, 0].set_ylim(-4, 4); axes[1, 0].set_aspect('equal')

colors = ['crimson', 'darkgreen', 'darkorange']
class_names = ['Moons', 'Circles', 'Gaussian']
for idx, (cls, color, name) in enumerate(zip([1, 2, 3], colors, class_names)):
    samples = cfg_sample(cond_model, cond_params, target_class=cls, 
                        n_samples=1500, guidance_scale=3.0, T=500).cpu()
    axes[1, idx+1].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=3, alpha=0.5, c=color)
    axes[1, idx+1].set_title(f'Class {cls} ({name}) | w = 3.0', fontsize=11)
    axes[1, idx+1].set_xlim(-4, 4); axes[1, idx+1].set_ylim(-4, 4)
    axes[1, idx+1].set_aspect('equal')

plt.suptitle('Classifier-Free Guidance: Controlling Generation with Guidance Scale w', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Key Insight: Higher guidance scale → samples cluster tighter")
print("around the conditional distribution (less diversity, more fidelity)")
print("="*60)

## 9. Advanced Topics

### 9.1 Consistency Models (Song et al., 2023)

**Industrial Example**: **SDXL Turbo** (Stability AI) and **LCM-LoRA** achieve high-quality image generation in just **1-4 steps** using distilled consistency models, enabling real-time generation on consumer GPUs.

**Core Idea**: Learn a function $$f_\theta(x_t, t)$$ that maps any point on the same ODE trajectory to the trajectory's origin $$x_0$$:

$$f_\theta(x_t, t) = f_\theta(x_{t'}, t') \quad \forall t, t' \in [0, T]$$

This "self-consistency" property means the model can jump from any noise level directly to a clean sample.

**Training Approaches**:
1. **Consistency Distillation (CD)**: Distill from a pre-trained diffusion model
   $$\mathcal{L}_{CD} = \mathbb{E}\left[d\left(f_\theta(x_{t_{n+1}}, t_{n+1}), f_{\theta^-}(\hat{x}_{t_n}, t_n)\right)\right]$$
   where $$\hat{x}_{t_n}$$ is obtained by one ODE step from $$x_{t_{n+1}}$$

2. **Consistency Training (CT)**: Train from scratch without a teacher
   $$\mathcal{L}_{CT} = \mathbb{E}\left[d\left(f_\theta(x + t_{n+1}\epsilon, t_{n+1}), f_{\theta^-}(x + t_n\epsilon, t_n)\right)\right]$$

### 9.2 Flow Matching (Lipman et al., 2023)

**Industrial Example**: **Stable Diffusion 3** (Stability AI) and **Meta's Movie Gen** use Rectified Flow / Flow Matching for improved training stability and sample quality.

**Core Idea**: Instead of noising and denoising, learn a velocity field that **transports** noise to data along straight paths:

$$\frac{dx_t}{dt} = v_\theta(x_t, t)$$

The interpolation path between noise $$x_1 \sim \mathcal{N}(0, I)$$ and data $$x_0 \sim p_{\text{data}}$$:

$$x_t = (1-t) x_0 + t x_1$$

The **conditional velocity** at time $$t$$:

$$u_t(x | x_0, x_1) = x_1 - x_0$$

**Flow Matching Loss** (simple regression on the velocity field):

$$\mathcal{L}_{FM} = \mathbb{E}_{t, x_0, x_1}\left[\| v_\theta(x_t, t) - (x_1 - x_0) \|^2\right]$$

**Advantages over standard diffusion**:
* Straighter trajectories → fewer sampling steps needed
* Simpler loss formulation (no noise schedules to tune)
* Naturally connects to optimal transport theory

### 9.3 Rectified Flow

**Refinement**: After initial training, "rectify" the flow by:
1. Generate pairs $$(x_0, x_1)$$ using the learned flow
2. Retrain on these pairs to straighten the trajectories further
3. Repeat for progressively straighter paths

After rectification, the paths become nearly straight, enabling 1-step generation.

In [0]:
# =============================================================================
# Flow Matching: Modern Alternative to Standard Diffusion
# Industrial Context: Used in Stable Diffusion 3, Meta Movie Gen
# =============================================================================

class FlowMatchingVelocityNet(nn.Module):
    """Velocity prediction network for Flow Matching.
    Learns v_theta(x_t, t) that transports noise to data."""
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, time_dim: int = 64):
        super().__init__()
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
        )
        self.net = nn.Sequential(
            nn.Linear(data_dim + time_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, data_dim),
        )
    
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t_emb = self.time_embed(t * 1000)
        return self.net(torch.cat([x, t_emb], dim=-1))


def train_flow_matching(model, data, n_epochs=200, lr=2e-3):
    """Train Flow Matching model.
    Loss: ||v_theta(x_t, t) - (x_1 - x_0)||^2
    where x_t = (1-t)*x_0 + t*x_1, x_1 ~ N(0,I)
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    
    for epoch in range(n_epochs):
        # Sample time uniformly
        t = torch.rand(data.shape[0], device=data.device)
        
        # Sample noise (x_1)
        x_1 = torch.randn_like(data)
        
        # Interpolate: x_t = (1-t)*x_0 + t*x_1
        x_t = (1 - t[:, None]) * data + t[:, None] * x_1
        
        # Target velocity: x_1 - x_0 (direction from data to noise)
        target_v = x_1 - data
        
        # Predict velocity
        v_pred = model(x_t, t)
        
        # MSE loss
        loss = F.mse_loss(v_pred, target_v)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
        
        if (epoch + 1) % 50 == 0:
            print(f"  Epoch {epoch+1}/{n_epochs} | Flow Matching Loss: {loss.item():.6f}")
    
    return losses


@torch.no_grad()
def flow_matching_sample(model, n_samples: int, data_dim: int = 2, 
                         n_steps: int = 50, device='cpu') -> torch.Tensor:
    """Generate samples by integrating the learned velocity field.
    Starting from noise (t=1), integrate backward to data (t=0).
    Using Euler method: x_{t-dt} = x_t - dt * v_theta(x_t, t)
    """
    model.eval()
    x = torch.randn(n_samples, data_dim, device=device)  # Start from noise
    dt = 1.0 / n_steps
    
    trajectory = [x.clone().cpu()]
    
    for step in range(n_steps):
        t = torch.full((n_samples,), 1.0 - step * dt, device=device)
        v = model(x, t)
        x = x - dt * v  # Euler integration (going from t=1 to t=0)
        trajectory.append(x.clone().cpu())
    
    return x, trajectory


# Train Flow Matching model
print("Training Flow Matching model...")
fm_model = FlowMatchingVelocityNet(data_dim=2, hidden_dim=256).to(device)
fm_losses = train_flow_matching(fm_model, data_2d.to(device), n_epochs=200)

# Sample with different step counts
print("\nSampling with Flow Matching (various steps)...")
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

# Row 1: Generated samples with different step counts
step_counts = [5, 10, 25, 100]
for idx, n_steps in enumerate(step_counts):
    fm_samples, _ = flow_matching_sample(fm_model, 2000, data_dim=2, n_steps=n_steps, device=device)
    fm_samples = fm_samples.cpu()
    axes[0, idx].scatter(fm_samples[:, 0].numpy(), fm_samples[:, 1].numpy(), s=2, alpha=0.5, c='darkviolet')
    axes[0, idx].set_title(f'Flow Matching ({n_steps} steps)', fontsize=11)
    axes[0, idx].set_xlim(-3.5, 3.5); axes[0, idx].set_ylim(-3.5, 3.5)
    axes[0, idx].set_aspect('equal')

# Row 2: Trajectory visualization (flow lines)
fm_samples_traj, trajectory = flow_matching_sample(fm_model, 50, data_dim=2, n_steps=30, device=device)

# Show the flow trajectories
axes[1, 0].scatter(data_2d[:, 0].numpy(), data_2d[:, 1].numpy(), s=1, alpha=0.2, c='steelblue')
axes[1, 0].set_title('Original Data', fontsize=11)
axes[1, 0].set_xlim(-3.5, 3.5); axes[1, 0].set_ylim(-3.5, 3.5); axes[1, 0].set_aspect('equal')

# Plot flow lines
trajectory_np = [t.numpy() for t in trajectory]
axes[1, 1].set_title('Flow Trajectories\n(Noise → Data)', fontsize=11)
for i in range(min(30, len(trajectory_np[0]))):
    xs = [trajectory_np[step][i, 0] for step in range(len(trajectory_np))]
    ys = [trajectory_np[step][i, 1] for step in range(len(trajectory_np))]
    axes[1, 1].plot(xs, ys, alpha=0.4, linewidth=0.8, c='darkviolet')
    axes[1, 1].scatter(xs[0], ys[0], s=10, c='red', zorder=5)  # Start (noise)
    axes[1, 1].scatter(xs[-1], ys[-1], s=10, c='green', zorder=5)  # End (data)
axes[1, 1].set_xlim(-4, 4); axes[1, 1].set_ylim(-4, 4); axes[1, 1].set_aspect('equal')

# Training loss comparison
axes[1, 2].plot(fm_losses, color='darkviolet', label='Flow Matching', linewidth=1.5)
axes[1, 2].plot(losses[:200], color='darkblue', label='DDPM', linewidth=1.5)
axes[1, 2].set_title('Loss Comparison', fontsize=11)
axes[1, 2].set_xlabel('Epoch')
axes[1, 2].set_yscale('log')
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

# Quality vs Steps comparison
axes[1, 3].set_title('Flow Matching Advantage:\nStraighter Paths = Fewer Steps', fontsize=11)
axes[1, 3].text(0.5, 0.7, 'DDPM: 1000 steps typical\nDDIM: 50 steps\nFlow Matching: 10-25 steps\nConsistency: 1-4 steps', 
               transform=axes[1, 3].transAxes, fontsize=12, va='center', ha='center',
               fontfamily='monospace', bbox=dict(boxstyle='round', facecolor='lightyellow'))
axes[1, 3].axis('off')

plt.suptitle('Flow Matching: Learning Straight Transport Paths', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9.4 Diffusion Transformers (DiT)

**Paper**: Peebles & Xie (2023) — "Scalable Diffusion Models with Transformers" (*ICCV*)

**Industrial Example**: DiT is the backbone of **OpenAI's Sora** (video generation), **Stable Diffusion 3** (Stability AI), **Flux** (Black Forest Labs), and **PixArt-α** (Huawei). It replaces the UNet with a Transformer, unlocking favorable scaling properties — larger models consistently produce better results.

---

### 9.4.1 Motivation: Why Replace the UNet?

The UNet has been the default architecture for diffusion models since DDPM, but it has limitations:

| Aspect | UNet | Transformer (DiT) |
| --- | --- | --- |
| Scaling behavior | Saturates at large compute | Smooth power-law scaling |
| Global context | Limited (local convolutions + sparse attention) | Full self-attention at every layer |
| Architecture flexibility | Complex skip connections, hand-tuned | Uniform blocks, easy to scale |
| Hardware efficiency | Irregular memory patterns | Optimized for modern GPUs/TPUs |
| Multi-modal conditioning | Bolted-on cross-attention | Native via adaptive normalization or joint attention |

**Key Result**: DiT-XL/2 achieves **FID 2.27** on class-conditional ImageNet 256×256, surpassing all prior UNet-based diffusion models while following predictable scaling laws (Gflops vs. FID is nearly linear on a log-log plot).

### 9.4.2 Architecture Overview

```
Input Image (or Latent)         Conditioning (class, text, time)
       │                                    │
       ↓                                    │
┌───────────────┐                           │
│  Patchify     │  (image → sequence of patches)  │
│  + Pos Embed  │                           │
└───────────────┘                           │
       │                                    │
       ↓                                    ↓
┌──────────────────────────────────────┐
│         DiT Block × N                     │
│  ┌──────────────────────────────────┐ │
│  │  AdaLN-Zero Modulation (γ, β, α)    │ │
│  │       (from time + class embed)     │ │
│  ├──────────────────────────────────┤ │
│  │  Multi-Head Self-Attention          │ │
│  ├──────────────────────────────────┤ │
│  │  Pointwise FeedForward (MLP)        │ │
│  └──────────────────────────────────┘ │
└──────────────────────────────────────┘
       │
       ↓
┌───────────────┐
│  Unpatchify   │  (sequence → spatial noise prediction)
│  (Linear)     │
└───────────────┘
       │
       ↓
  Predicted Noise ε_θ (or v, x_0)
```

### 9.4.3 Patchification: Images as Token Sequences

An image (or latent) of size $$H \times W \times C$$ is divided into non-overlapping patches of size $$p \times p$$, creating a sequence of length:

$$N = \frac{H \times W}{p^2}$$

Each patch is linearly projected to dimension $$d$$, then position embeddings are added:

$$z_0 = [\text{Linear}(\text{patch}_1), \ldots, \text{Linear}(\text{patch}_N)] + E_{\text{pos}}$$

**Patch size controls the compute-quality tradeoff**:
* DiT-XL/2: $$p=2$$ (finest, most expensive, best quality) → $$32 \times 32 = 1024$$ tokens for $$64 \times 64$$ latents
* DiT-XL/4: $$p=4$$ (coarser, $$4\times$$ faster) → $$16 \times 16 = 256$$ tokens
* DiT-XL/8: $$p=8$$ (coarsest) → $$8 \times 8 = 64$$ tokens

### 9.4.4 Conditioning via Adaptive Layer Norm (adaLN-Zero)

The critical design choice: how to inject timestep $$t$$ and class/text conditioning $$c$$ into the Transformer blocks.

**adaLN-Zero** (best performing): Learn scale ($$\gamma$$), shift ($$\beta$$), and gate ($$\alpha$$) parameters from the conditioning:

$$(\gamma_1, \beta_1, \alpha_1, \gamma_2, \beta_2, \alpha_2) = \text{MLP}(t_{\text{emb}} + c_{\text{emb}})$$

Applied as:
$$h = x + \alpha_1 \odot \text{Attention}(\gamma_1 \odot \text{LN}(x) + \beta_1)$$
$$\text{out} = h + \alpha_2 \odot \text{FFN}(\gamma_2 \odot \text{LN}(h) + \beta_2)$$

The "Zero" refers to initializing $$\alpha$$ to zero, making each block an identity function at initialization (crucial for training stability at scale).

**Alternative conditioning approaches** explored in the paper:

| Method | Description | FID |
| --- | --- | --- |
| In-context | Append $$t$$ and $$c$$ as extra tokens | 5.28 |
| Cross-attention | Condition via cross-attention (like UNet) | 4.21 |
| adaLN | Adaptive LayerNorm (scale + shift) | 3.01 |
| **adaLN-Zero** | adaLN + learnable gating (initialized at 0) | **2.27** |

### 9.4.5 Scaling Laws

The DiT paper demonstrates clear scaling behavior:

| Model | Depth | Width | Params | Gflops | FID-50K↓ |
| --- | --- | --- | --- | --- | --- |
| DiT-S/2 | 12 | 384 | 33M | 6 | 68.4 |
| DiT-B/2 | 12 | 768 | 130M | 24 | 43.5 |
| DiT-L/2 | 24 | 1024 | 458M | 80 | 9.62 |
| DiT-XL/2 | 28 | 1152 | 675M | 119 | **2.27** |

This log-linear relationship between compute (Gflops) and quality (FID) mirrors the scaling laws observed in large language models (Chinchilla, GPT-4).

### 9.4.6 DiT Variants in Production

| System | DiT Variant | Key Modification |
| --- | --- | --- |
| Stable Diffusion 3 | MM-DiT | Joint attention over image + text tokens |
| Flux.1 | Single-stream MM-DiT | Concatenated modalities, single attention |
| Sora | Spacetime DiT | 3D patches (height × width × time) |
| PixArt-α | Cross-attn DiT | Efficient T5-conditioned generation |
| Hunyuan-DiT | Dual-stream DiT | Bilingual text encoders |

### 9.4.7 MM-DiT: Joint Attention (Stable Diffusion 3)

Multi-Modal DiT concatenates image and text tokens into a single sequence and runs joint self-attention:

$$[z_{\text{img}}; z_{\text{txt}}] = \text{SelfAttention}([\text{patches}; \text{text\_tokens}])$$

This allows bidirectional information flow between modalities (text tokens attend to image and vice versa), producing tighter text-image alignment than separate cross-attention.

In [0]:
# =============================================================================
# Diffusion Transformer (DiT) Architecture Implementation
# Industrial Context: Backbone of Sora, Stable Diffusion 3, Flux
# =============================================================================

# --- Adaptive Layer Norm with Zero-Init Gating (adaLN-Zero) ---
class AdaLNZero(nn.Module):
    """Adaptive Layer Normalization with zero-initialized gating.
    The key conditioning mechanism in DiT.
    
    Learns (gamma, beta, alpha) from conditioning signal:
    - gamma, beta: scale and shift the normalized activations
    - alpha: gate the output (initialized to 0 for identity at init)
    """
    def __init__(self, hidden_dim: int, cond_dim: int):
        super().__init__()
        self.norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        # Produces 6 modulation parameters: (gamma1, beta1, alpha1, gamma2, beta2, alpha2)
        self.adaLN_modulation = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 6 * hidden_dim),
        )
        # Zero-initialize the output projection for identity initialization
        nn.init.zeros_(self.adaLN_modulation[-1].weight)
        nn.init.zeros_(self.adaLN_modulation[-1].bias)
    
    def forward(self, conditioning: torch.Tensor) -> tuple:
        """Returns 6 modulation vectors split into pairs for attention and FFN."""
        modulation = self.adaLN_modulation(conditioning)
        return modulation.chunk(6, dim=-1)


# --- Multi-Head Self-Attention ---
class MultiHeadSelfAttention(nn.Module):
    """Standard multi-head self-attention (core of the Transformer)."""
    def __init__(self, hidden_dim: int, n_heads: int = 8):
        super().__init__()
        self.n_heads = n_heads
        self.head_dim = hidden_dim // n_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.proj = nn.Linear(hidden_dim, hidden_dim)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)  # Each: [B, heads, N, head_dim]
        
        # Scaled dot-product attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        return self.proj(x)


# --- DiT Block ---
class DiTBlock(nn.Module):
    """A single Diffusion Transformer block.
    Consists of: adaLN-modulated self-attention + adaLN-modulated FFN.
    
    Key difference from standard Transformer:
    - No cross-attention (conditioning is via adaLN modulation)
    - Zero-initialized output gates (alpha) for training stability
    """
    def __init__(self, hidden_dim: int, n_heads: int = 8, mlp_ratio: float = 4.0, cond_dim: int = 128):
        super().__init__()
        self.norm1 = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.attn = MultiHeadSelfAttention(hidden_dim, n_heads)
        self.norm2 = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        
        mlp_hidden = int(hidden_dim * mlp_ratio)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, mlp_hidden),
            nn.GELU(),
            nn.Linear(mlp_hidden, hidden_dim),
        )
        
        # adaLN-Zero: produces modulation parameters from conditioning
        self.adaLN = AdaLNZero(hidden_dim, cond_dim)
    
    def forward(self, x: torch.Tensor, conditioning: torch.Tensor) -> torch.Tensor:
        # Get all 6 modulation parameters
        gamma1, beta1, alpha1, gamma2, beta2, alpha2 = self.adaLN(conditioning)
        
        # Self-attention branch with adaLN modulation
        h = self.norm1(x)
        h = h * (1 + gamma1.unsqueeze(1)) + beta1.unsqueeze(1)  # Modulate
        h = self.attn(h)
        x = x + alpha1.unsqueeze(1) * h  # Gated residual
        
        # FFN branch with adaLN modulation
        h = self.norm2(x)
        h = h * (1 + gamma2.unsqueeze(1)) + beta2.unsqueeze(1)  # Modulate
        h = self.ffn(h)
        x = x + alpha2.unsqueeze(1) * h  # Gated residual
        
        return x


# --- Full DiT Model ---
class DiT(nn.Module):
    """Diffusion Transformer for 2D data (simplified).
    
    In production (DiT-XL/2 for ImageNet 256x256):
    - Input: 64x64x4 latent (from VAE encoder)
    - Patch size: 2x2 -> 32x32 = 1024 tokens
    - Hidden dim: 1152, Depth: 28, Heads: 16
    - Parameters: 675M
    
    Here we demonstrate the architecture on 2D point data,
    treating each 2D point as a single "patch" (token).
    """
    def __init__(self, data_dim: int = 2, hidden_dim: int = 256, 
                 depth: int = 6, n_heads: int = 8, n_classes: int = 3,
                 time_dim: int = 128):
        super().__init__()
        self.data_dim = data_dim
        cond_dim = time_dim
        
        # Input projection (analogous to patchify + linear projection)
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        
        # Timestep embedding
        self.time_embed = nn.Sequential(
            SinusoidalPositionEmbedding(time_dim),
            nn.Linear(time_dim, cond_dim),
            nn.SiLU(),
            nn.Linear(cond_dim, cond_dim),
        )
        
        # Class embedding (label conditioning)
        self.class_embed = nn.Embedding(n_classes + 1, cond_dim)  # +1 for null class
        
        # Transformer blocks
        self.blocks = nn.ModuleList([
            DiTBlock(hidden_dim, n_heads, mlp_ratio=4.0, cond_dim=cond_dim)
            for _ in range(depth)
        ])
        
        # Final layer: adaLN + linear projection to noise
        self.final_norm = nn.LayerNorm(hidden_dim, elementwise_affine=False)
        self.final_adaLN = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, 2 * hidden_dim),  # gamma, beta for final norm
        )
        self.output_proj = nn.Linear(hidden_dim, data_dim)
        
        # Zero-init the output projection (like adaLN-Zero)
        nn.init.zeros_(self.output_proj.weight)
        nn.init.zeros_(self.output_proj.bias)
    
    def forward(self, x: torch.Tensor, t: torch.Tensor, 
                class_label: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: Noisy data [B, data_dim]
            t: Timesteps [B]
            class_label: Class labels [B] (0 = unconditional)
        Returns:
            Predicted noise [B, data_dim]
        """
        # Compute conditioning: time + class
        t_emb = self.time_embed(t.float())
        c_emb = self.class_embed(class_label)
        conditioning = t_emb + c_emb  # [B, cond_dim]
        
        # Project input to hidden dim and add sequence dimension
        h = self.input_proj(x).unsqueeze(1)  # [B, 1, hidden_dim]
        
        # Pass through DiT blocks
        for block in self.blocks:
            h = block(h, conditioning)
        
        # Final normalization with adaLN
        final_mod = self.final_adaLN(conditioning)
        gamma, beta = final_mod.chunk(2, dim=-1)
        h = self.final_norm(h)
        h = h * (1 + gamma.unsqueeze(1)) + beta.unsqueeze(1)
        
        # Project to output
        output = self.output_proj(h).squeeze(1)  # [B, data_dim]
        return output


# --- Train DiT on the class-conditional data ---
print("="*60)
print("DIFFUSION TRANSFORMER (DiT): Training")
print("="*60)

dit_model = DiT(
    data_dim=2, hidden_dim=256, depth=6, n_heads=8, n_classes=3, time_dim=128
).to(device)

print(f"\nDiT Architecture:")
print(f"  Depth: 6 blocks")
print(f"  Hidden dim: 256")
print(f"  Attention heads: 8")
print(f"  Parameters: {sum(p.numel() for p in dit_model.parameters()):,}")
print(f"  Conditioning: adaLN-Zero (time + class)")

dit_optimizer = torch.optim.AdamW(dit_model.parameters(), lr=1e-3, weight_decay=0.01)

# Use cosine schedule
dit_betas = cosine_beta_schedule(500).to(device)
dit_params = compute_schedule_params(dit_betas)
for k, v in dit_params.items():
    dit_params[k] = v.to(device)

# Training with CFG dropout (10% unconditional)
cond_drop_prob = 0.1
T_dit = 500

print("\nTraining DiT with Classifier-Free Guidance (p_uncond=0.1)...")
dit_losses = []

for epoch in range(200):
    idx = torch.randperm(class_data.shape[0], device=device)[:512]
    x_0 = class_data[idx].to(device)
    labels = class_labels[idx].to(device)
    
    # CFG: randomly drop labels
    drop_mask = torch.rand(512, device=device) < cond_drop_prob
    labels_with_drop = labels.clone()
    labels_with_drop[drop_mask] = 0  # 0 = null/unconditional
    
    # Standard diffusion training
    t = torch.randint(0, T_dit, (512,), device=device)
    noise = torch.randn_like(x_0)
    x_t = dit_params['sqrt_alphas_cumprod'][t][:, None] * x_0 + \
          dit_params['sqrt_one_minus_alphas_cumprod'][t][:, None] * noise
    
    noise_pred = dit_model(x_t, t, labels_with_drop)
    loss = F.mse_loss(noise_pred, noise)
    
    dit_optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(dit_model.parameters(), 1.0)
    dit_optimizer.step()
    dit_losses.append(loss.item())
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/200 | Loss: {loss.item():.6f}")


# --- Sample from DiT with CFG ---
@torch.no_grad()
def dit_cfg_sample(model, params, target_class: int, n_samples: int = 1000,
                   guidance_scale: float = 4.0, T: int = 500) -> torch.Tensor:
    """Sample from DiT using Classifier-Free Guidance."""
    model.eval()
    x = torch.randn(n_samples, 2, device=device)
    labels = torch.full((n_samples,), target_class, device=device, dtype=torch.long)
    null_labels = torch.zeros(n_samples, device=device, dtype=torch.long)
    
    for t_val in reversed(range(T)):
        t_batch = torch.full((n_samples,), t_val, device=device, dtype=torch.long)
        
        # Conditional and unconditional predictions
        eps_cond = model(x, t_batch, labels)
        eps_uncond = model(x, t_batch, null_labels)
        
        # CFG
        eps = eps_uncond + guidance_scale * (eps_cond - eps_uncond)
        
        # Denoise
        beta = params['betas'][t_val]
        sqrt_recip_alpha = params['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_ab = params['sqrt_one_minus_alphas_cumprod'][t_val]
        
        mean = sqrt_recip_alpha * (x - (beta / sqrt_one_minus_ab) * eps)
        
        if t_val > 0:
            sigma = torch.sqrt(params['posterior_variance'][t_val])
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean
    
    return x


# Generate and visualize
print("\nGenerating samples from DiT with CFG...")
fig, axes = plt.subplots(1, 5, figsize=(25, 5))

# Training data
axes[0].scatter(class_data[:, 0].numpy(), class_data[:, 1].numpy(),
               s=2, alpha=0.3, c=class_labels.numpy(), cmap='Set1')
axes[0].set_title('Training Data (3 classes)', fontsize=12)
axes[0].set_xlim(-4, 4); axes[0].set_ylim(-4, 4); axes[0].set_aspect('equal')

# Per-class generation with CFG
colors = ['crimson', 'darkgreen', 'darkorange']
class_names = ['Moons', 'Circles', 'Gaussian']
for idx, (cls, color, name) in enumerate(zip([1, 2, 3], colors, class_names)):
    samples = dit_cfg_sample(dit_model, dit_params, target_class=cls,
                            n_samples=1500, guidance_scale=4.0, T=500).cpu()
    axes[idx+1].scatter(samples[:, 0].numpy(), samples[:, 1].numpy(), s=3, alpha=0.5, c=color)
    axes[idx+1].set_title(f'DiT: Class {cls} ({name})\nw=4.0', fontsize=11)
    axes[idx+1].set_xlim(-4, 4); axes[idx+1].set_ylim(-4, 4)
    axes[idx+1].set_aspect('equal')

# Training curve
axes[4].plot(dit_losses, color='navy', linewidth=1.5)
axes[4].set_title('DiT Training Loss', fontsize=12)
axes[4].set_xlabel('Epoch')
axes[4].set_yscale('log')
axes[4].grid(True, alpha=0.3)

plt.suptitle('Diffusion Transformer (DiT): Class-Conditional Generation with adaLN-Zero', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("DiT Key Takeaways:")
print("  • Replaces UNet with Transformer: simpler, scales better")
print("  • adaLN-Zero conditioning: time+class modulate every layer")
print("  • Zero-initialization of gates: identity at init, stable training")
print("  • Scaling law: more compute → predictably better FID")
print("  • Powers: Sora, SD3, Flux, PixArt-α, Hunyuan")
print("="*60)

## 10. Industrial Applications of Diffusion Models

Diffusion models have become the dominant generative paradigm across industries. Below are detailed applications with the specific techniques employed.

---

### 10.1 Image Generation & Editing

| Product | Company | Architecture | Key Innovation |
| --- | --- | --- | --- |
| Stable Diffusion 3 | Stability AI | LDM + Flow Matching + DiT | Rectified Flow for faster sampling |
| DALL·E 3 | OpenAI | Cascaded LDM | Improved text-image alignment via caption finetuning |
| Midjourney v6 | Midjourney | Proprietary LDM | Aesthetic fine-tuning + style control |
| Imagen 3 | Google DeepMind | Cascaded pixel-space + T5-XXL | Largest text encoder for understanding |
| Adobe Firefly | Adobe | LDM + ControlNet | Commercially safe training data |

**Techniques Used**: DDPM/DDIM backbone, Classifier-Free Guidance ($$w = 7.5$$), Cross-Attention conditioning, ControlNet for structural guidance

### 10.2 Video Generation

| Product | Company | Approach |
| --- | --- | --- |
| Sora | OpenAI | Spacetime patches + DiT in latent space |
| Movie Gen | Meta | Flow Matching + temporal attention |
| Runway Gen-3 | Runway | Temporal LDM with motion prediction |
| Stable Video Diffusion | Stability AI | Image-conditioned temporal diffusion |

**Key Extension**: Add temporal attention layers to the UNet/DiT and model video as a 3D latent (height × width × time).

### 10.3 Audio & Music

| Product | Company | Application |
| --- | --- | --- |
| AudioLDM 2 | Stability AI | Text-to-audio + music generation |
| MusicGen | Meta | Music generation from text prompts |
| Bark | Suno | Text-to-speech with prosody control |
| Riffusion | - | Real-time music via spectrogram diffusion |

**Technique**: Apply diffusion to mel-spectrograms (2D representation of audio), then use a vocoder (HiFi-GAN) to convert back to waveform.

### 10.4 Drug Discovery & Molecular Generation

| Application | Approach | Company/Lab |
| --- | --- | --- |
| Molecular conformation | SE(3)-equivariant diffusion | Generate:Biomedicines |
| Protein structure | Diffusion on $$\text{SO}(3)$$ manifold | RFdiffusion (Baker Lab) |
| Drug binding pose | Score-based on SE(3) | DiffDock (MIT) |
| Retrosynthesis | Graph diffusion | Multiple labs |

**Key Innovation**: Diffusion on non-Euclidean spaces (rotation groups, manifolds) with equivariant architectures.

$$\text{RFdiffusion}$$: Designs novel protein structures by denoising residue frames on $$\text{SE}(3)^N$$ (the product of rigid-body transformations).

### 10.5 3D Generation

| Product | Company | Method |
| --- | --- | --- |
| Point-E / Shap-E | OpenAI | Point cloud / implicit function diffusion |
| GET3D | NVIDIA | SDE-based texture + geometry generation |
| DreamFusion | Google | Score distillation from 2D diffusion to NeRF |
| Magic3D | NVIDIA | Two-stage: coarse NeRF + fine mesh |

**Score Distillation Sampling (SDS)**: Use a pre-trained 2D diffusion model as a critic to optimize a 3D representation:

$$\nabla_\phi \mathcal{L}_{SDS} = \mathbb{E}_{t, \epsilon}\left[w(t)(\epsilon_\theta(x_t; t, c) - \epsilon) \frac{\partial g(\phi)}{\partial \phi}\right]$$

where $$g(\phi)$$ renders the 3D representation from a random viewpoint.

### 10.6 Robotics & Planning

| Application | Method | Lab |
| --- | --- | --- |
| Robot action planning | Diffusion Policy | Columbia/Toyota |
| Trajectory optimization | Diffuser | UC Berkeley |
| Dexterous manipulation | SE(3) diffusion | CMU |

**Diffusion Policy**: Model the robot's action distribution as a diffusion process conditioned on observations:
$$\pi(a_t | o_t) \sim p_\theta(a_t | o_t) \text{ via DDPM denoising}$$

Handles multi-modal action distributions (multiple valid actions for same observation) naturally.

### 10.7 Time Series & Tabular Data

| Application | Method | Use Case |
| --- | --- | --- |
| TimeGrad | Autoregressive diffusion | Probabilistic forecasting |
| CSDI | Conditional Score-based | Imputation + forecasting |
| TabDDPM | DDPM for tabular | Synthetic data generation |
| DiffusionTS | Transformer diffusion | Long-horizon forecasting |

**Industrial Value**: Generate synthetic tabular data for privacy-preserving analytics, data augmentation for rare-event prediction (fraud, equipment failure).

In [0]:
# =============================================================================
# Application: Diffusion Model for Time Series Generation
# Industrial Context: Synthetic data generation, probabilistic forecasting
# (TabDDPM, TimeGrad, CSDI patterns)
# =============================================================================

class TimeSeriesDiffusion:
    """Diffusion model for 1D time series generation.
    Demonstrates the core idea behind TimeGrad and CSDI.
    
    Industrial Use Cases:
    - Synthetic financial data generation (privacy compliance)
    - Probabilistic demand forecasting
    - Anomaly detection (model normal behavior, flag deviations)
    - Data augmentation for rare events (fraud, equipment failure)
    """
    def __init__(self, seq_len: int = 50, hidden_dim: int = 128, T: int = 200):
        self.seq_len = seq_len
        self.T = T
        self.model = self._build_model(seq_len, hidden_dim)
        self.betas = cosine_beta_schedule(T)
        self.params = compute_schedule_params(self.betas)
    
    def _build_model(self, seq_len, hidden_dim):
        """Simple temporal model (in production: Transformer or WaveNet)."""
        return nn.Sequential(
            nn.Linear(seq_len + 64, hidden_dim),  # Input + time embedding
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, seq_len),
        )


# Generate synthetic time series data (e.g., stock-like patterns)
def generate_synthetic_timeseries(n_samples: int = 2000, seq_len: int = 50) -> torch.Tensor:
    """Generate diverse time series patterns mimicking financial/sensor data."""
    series = []
    for _ in range(n_samples):
        pattern = np.random.choice(['trend', 'seasonal', 'mixed'])
        t = np.linspace(0, 4*np.pi, seq_len)
        
        if pattern == 'trend':
            s = np.cumsum(np.random.randn(seq_len) * 0.3) + np.random.randn() * 2
        elif pattern == 'seasonal':
            freq = np.random.uniform(0.5, 2.0)
            amp = np.random.uniform(0.5, 2.0)
            s = amp * np.sin(freq * t) + np.random.randn(seq_len) * 0.2
        else:  # mixed
            trend = np.linspace(0, np.random.randn() * 2, seq_len)
            seasonal = np.random.uniform(0.5, 1.5) * np.sin(np.random.uniform(0.5, 1.5) * t)
            s = trend + seasonal + np.random.randn(seq_len) * 0.2
        
        series.append(s)
    
    data = torch.tensor(np.array(series), dtype=torch.float32)
    # Normalize
    data = (data - data.mean()) / data.std()
    return data


# Generate training data
ts_data = generate_synthetic_timeseries(n_samples=3000, seq_len=50).to(device)
print(f"Time series dataset: {ts_data.shape} (samples x sequence_length)")

# Build and train model
ts_model = nn.Sequential(
    nn.Linear(50 + 64, 256),
    nn.SiLU(),
    nn.Linear(256, 256),
    nn.SiLU(),
    nn.Linear(256, 256),
    nn.SiLU(),
    nn.Linear(256, 50),
).to(device)

time_embed_ts = SinusoidalPositionEmbedding(64).to(device)

betas_ts = cosine_beta_schedule(200).to(device)
params_ts = compute_schedule_params(betas_ts)
for k, v in params_ts.items():
    params_ts[k] = v.to(device)

optimizer_ts = torch.optim.Adam(ts_model.parameters(), lr=2e-4)

print("\nTraining Time Series Diffusion Model...")
for epoch in range(200):
    idx = torch.randperm(ts_data.shape[0])[:512]
    x_0 = ts_data[idx]
    
    t = torch.randint(0, 200, (512,), device=device)
    noise = torch.randn_like(x_0)
    x_t = params_ts['sqrt_alphas_cumprod'][t][:, None] * x_0 + \
          params_ts['sqrt_one_minus_alphas_cumprod'][t][:, None] * noise
    
    t_emb = time_embed_ts(t.float())
    inp = torch.cat([x_t, t_emb], dim=-1)
    noise_pred = ts_model(inp)
    
    loss = F.mse_loss(noise_pred, noise)
    optimizer_ts.zero_grad()
    loss.backward()
    optimizer_ts.step()
    
    if (epoch + 1) % 50 == 0:
        print(f"  Epoch {epoch+1}/200 | Loss: {loss.item():.6f}")

# Generate time series samples
print("\nGenerating synthetic time series...")
ts_model.eval()
with torch.no_grad():
    x = torch.randn(100, 50, device=device)
    for t_val in reversed(range(200)):
        t_batch = torch.full((100,), t_val, device=device, dtype=torch.long)
        t_emb = time_embed_ts(t_batch.float())
        inp = torch.cat([x, t_emb], dim=-1)
        eps_pred = ts_model(inp)
        
        beta = params_ts['betas'][t_val]
        sqrt_recip_alpha = params_ts['sqrt_recip_alphas'][t_val]
        sqrt_one_minus_ab = params_ts['sqrt_one_minus_alphas_cumprod'][t_val]
        
        mean = sqrt_recip_alpha * (x - (beta / sqrt_one_minus_ab) * eps_pred)
        
        if t_val > 0:
            sigma = torch.sqrt(params_ts['posterior_variance'][t_val])
            x = mean + sigma * torch.randn_like(x)
        else:
            x = mean

generated_ts = x.cpu().numpy()

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot real samples
for i in range(20):
    axes[0, 0].plot(ts_data[i].cpu().numpy(), alpha=0.5, linewidth=0.8)
axes[0, 0].set_title('Real Time Series (Training Data)', fontsize=12)
axes[0, 0].set_xlabel('Time Step')
axes[0, 0].set_ylabel('Value')

# Plot generated samples
for i in range(20):
    axes[0, 1].plot(generated_ts[i], alpha=0.5, linewidth=0.8)
axes[0, 1].set_title('Generated Time Series (Diffusion)', fontsize=12)
axes[0, 1].set_xlabel('Time Step')
axes[0, 1].set_ylabel('Value')

# Distribution comparison (per-timestep statistics)
real_mean = ts_data[:100].cpu().numpy().mean(axis=0)
real_std = ts_data[:100].cpu().numpy().std(axis=0)
gen_mean = generated_ts.mean(axis=0)
gen_std = generated_ts.std(axis=0)

axes[1, 0].plot(real_mean, label='Real Mean', color='blue', linewidth=2)
axes[1, 0].fill_between(range(50), real_mean - real_std, real_mean + real_std, alpha=0.2, color='blue')
axes[1, 0].plot(gen_mean, label='Generated Mean', color='red', linewidth=2, linestyle='--')
axes[1, 0].fill_between(range(50), gen_mean - gen_std, gen_mean + gen_std, alpha=0.2, color='red')
axes[1, 0].legend()
axes[1, 0].set_title('Statistical Comparison (Mean ± Std)', fontsize=12)
axes[1, 0].set_xlabel('Time Step')

# Autocorrelation comparison
from numpy import correlate as np_correlate

def autocorrelation(x, max_lag=20):
    x_centered = x - x.mean()
    result = np.correlate(x_centered, x_centered, mode='full')
    result = result[len(result)//2:len(result)//2 + max_lag]
    return result / result[0]

real_acf = np.mean([autocorrelation(ts_data[i].cpu().numpy()) for i in range(100)], axis=0)
gen_acf = np.mean([autocorrelation(generated_ts[i]) for i in range(100)], axis=0)

axes[1, 1].plot(real_acf, label='Real ACF', color='blue', linewidth=2)
axes[1, 1].plot(gen_acf, label='Generated ACF', color='red', linewidth=2, linestyle='--')
axes[1, 1].legend()
axes[1, 1].set_title('Autocorrelation Function Comparison', fontsize=12)
axes[1, 1].set_xlabel('Lag')

plt.suptitle('Diffusion for Time Series: Synthetic Data Generation', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Industrial Applications of Time Series Diffusion:")
print("  • Finance: Generate synthetic market data for strategy backtesting")
print("  • Healthcare: Augment rare disease progression patterns")
print("  • Manufacturing: Create synthetic sensor data for predictive maintenance")
print("  • Privacy: Share realistic data without exposing individual records")
print("="*60)

## 11. Summary & Comparative Analysis

### Algorithm Comparison Matrix

| Algorithm | Sampling Steps | Training Stability | Likelihood | Speed | Quality |
| --- | --- | --- | --- | --- | --- |
| DDPM | 1000 | Excellent | ELBO | Slow | High |
| Score-Based (NCSN) | 100-1000 | Good | No | Slow | High |
| Score SDE (VP/VE) | 100-2000 | Excellent | Exact (ODE) | Medium | Highest |
| DDIM | 10-100 | N/A (uses DDPM) | Approx | Fast | Good |
| Latent Diffusion | 20-50 | Excellent | ELBO | Fast | High |
| Flow Matching | 10-50 | Excellent | Exact (ODE) | Fast | High |
| Consistency Models | 1-4 | Moderate | No | Fastest | Good |

### Key Relationships

```
                    DDPM (Ho et al., 2020)
                   /         |         \
                  /          |          \
    Score-Based (Song 2019)  |   Improved DDPM (Nichol 2021)
                  \          |          /
                   \         |         /
              Score SDE (Song et al., 2021)
              /              |              \
             /               |               \
    VP-SDE (=DDPM)    VE-SDE (=NCSN)    Prob. Flow ODE
                             |                  |
                             |                  |
                    Flow Matching         DDIM (Song 2021)
                    (Lipman 2023)               |
                             |           Consistency Models
                             |           (Song 2023)
                    Stable Diffusion 3
```

### The Unified View (Key Takeaway)

All diffusion models share three ingredients:
1. A **corruption process** that maps data to noise (forward SDE / interpolation path)
2. A **neural network** that predicts either noise ($$\epsilon$$), score ($$\nabla_x \log p$$), data ($$x_0$$), or velocity ($$v$$)
3. An **iterative procedure** to reverse the corruption (reverse SDE / ODE integration)

The differences lie in:
* How the corruption is parameterized (discrete vs continuous, VP vs VE)
* What the network predicts (all are mathematically equivalent)
* How the reverse process is solved (stochastic vs deterministic, number of steps)

---

## 12. References

### Foundational Papers
1. Sohl-Dickstein et al. (2015). "Deep Unsupervised Learning using Nonequilibrium Thermodynamics." *ICML*.
2. Ho, Jain, Abbeel (2020). "Denoising Diffusion Probabilistic Models." *NeurIPS*.
3. Song & Ermon (2019). "Generative Modeling by Estimating Gradients of the Data Distribution." *NeurIPS*.
4. Song et al. (2021). "Score-Based Generative Modeling through Stochastic Differential Equations." *ICLR*.

### Accelerated Sampling
5. Song, Meng, Ermon (2021). "Denoising Diffusion Implicit Models." *ICLR*.
6. Lu et al. (2022). "DPM-Solver: A Fast ODE Solver for Diffusion Probabilistic Model Sampling." *NeurIPS*.
7. Song et al. (2023). "Consistency Models." *ICML*.

### Architecture & Conditioning
8. Rombach et al. (2022). "High-Resolution Image Synthesis with Latent Diffusion Models." *CVPR*.
9. Ho & Salimans (2022). "Classifier-Free Diffusion Guidance." *NeurIPS Workshop*.
10. Dhariwal & Nichol (2021). "Diffusion Models Beat GANs on Image Synthesis." *NeurIPS*.

### Modern Advances
11. Lipman et al. (2023). "Flow Matching for Generative Modeling." *ICLR*.
12. Liu et al. (2023). "Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow." *ICLR*.
13. Peebles & Xie (2023). "Scalable Diffusion Models with Transformers (DiT)." *ICCV*.

### Applications
14. Watson et al. (2023). "De novo design of protein structure and function with RFdiffusion." *Nature*.
15. Corso et al. (2023). "DiffDock: Diffusion Steps, Twists, and Turns for Molecular Docking." *ICLR*.
16. Chi et al. (2023). "Diffusion Policy: Visuomotor Policy Learning via Action Diffusion." *RSS*.
17. Rasul et al. (2021). "Autoregressive Denoising Diffusion Models for Multivariate Probabilistic Time Series Forecasting." *ICML*.